# 저항 점용접(RSW) 공정 최적화 — 줄열(Joule Heating) 기반 물리 피팅 파이프라인

## A-1 — 왜 이 주제를 다루는가

저항 점용접(Resistance Spot Welding, RSW)은 자동차 차체 조립 등 제조 현장에서 로봇이 가장 많이 반복
수행하는 공정 중 하나입니다. 그런데 실제 현장에서는 전류·통전시간·가압력 같은 공정 변수를 얼마나
정밀하게 맞추느냐에 따라 같은 로봇, 같은 부품이라도 **완전히 접합되지 않는 미융착(Bad)** 이나
**과도한 열로 금속이 튀는 팽출(Explode)** 같은 결함이 발생합니다. 결함은 사후 검사로 걸러낼 수도
있지만, 그보다 더 좋은 방향은 로봇이 용접을 수행하기 **직전에** "이 조건이 안전한가"를 판단할 수 있게
만드는 것입니다. 이 노트북은 "AI 기반 용접 로봇 소프트웨어"라는 문제의식에서 출발해, 실측 데이터로부터
이러한 판단 기준을 물리적으로 근거 있게 도출하는 과정을 처음부터 끝까지 다룹니다.

## A-2 — 무엇을, 어떻게 풀 것인가

접근 방식에서 가장 중요한 원칙은 하나입니다. **개별 데이터 포인트 하나하나를 예측하려 하지 않고,
구간별 분포(평균 ± 표준오차)에 물리 법칙 기반 비선형 곡선을 피팅한다**는 것입니다. 단일 포인트
예측은 노이즈에 취약하고 "왜 이 값인가"에 답하기 어렵지만, 분포 기반 피팅은 오차 범위(신뢰구간)를
함께 제공하므로 "이 결론을 얼마나 믿을 수 있는가"까지 정량적으로 답할 수 있습니다. 이 노트북 전체에서
반복해서 등장하는 **오차비율(오차/추정값, 30% 미만이면 "신뢰 가능")** 판정 기준이 그 실천입니다.

다루는 결함은 발생 메커니즘이 서로 다른 두 종류입니다.

| 결함 유형 | 열입력과의 관계 | 다루는 위치 |
|---|---|---|
| `Bad` (미융착) | 열입력이 낮을수록 급증 → 줄열 부족 | B-1c/D-1 (하한 피팅) |
| `Explode` (스패터/팽출) | 단변량 요인 자체가 불확실 | B-5 (요인 탐색 후 상한 피팅) |

핵심 물리 지배식은 줄열(Joule Heating)입니다.

$$Q = I^2 \cdot R \cdot t$$

접촉저항 $R$이 데이터셋에 직접 계측되어 있지 않으므로, 동일 모재(AISI 1010 carbon steel) 가정 하에
**$R=1$로 정규화한 상대적 열입력 proxy** $Q \propto I^2 t$ 를 사용합니다. 절대 물리량이 아니라
"조건 A가 조건 B보다 열을 더 많이/적게 받았는가"를 비교하기 위한 상대 지표라는 점이 이후 모든 해석의
전제가 됩니다.

## A-3 — 이 노트북에서 실제로 확인하게 되는 것

이 물리 기반 접근을 실측 493개 샘플에 적용하면, 처음에는 예상하지 못했던 문제들이 드러납니다. 예를
들어 구간을 나눠서 평균만으로 피팅했을 때는 파라미터 오차가 추정값의 87배까지 발산하는 현상이
나타났고, 원인을 추적한 끝에 "구간화 이전의 원본 샘플 전체에 직접 피팅해야 한다"는 결론에 이르렀습니다
(자세한 경위는 F 섹션 참고). 또한 팽출(Explode) 결함처럼 두 값만 갖는 범주형 변수에 연속 로지스틱을
강제로 피팅하면 수학적으로 답이 유일하게 정해지지 않는다는 것도 실제로 시도해보고 나서야 확인할 수
있었습니다. 이런 시행착오와 수정 과정 자체가 "물리 기반 피팅이 왜 신중해야 하는가"를 보여주는
사례이기도 합니다.

## A-4 — 이 노트북을 읽는 방법

아래 6가지 관점을 모두 다루도록 구성했습니다. **물리적 셀 순서는 실행 의존성(뒤 섹션이 앞 섹션의
변수·CSV를 재사용)을 지키기 위해 아래 나열 순서와 다소 다릅니다** — 실제 순서는 A(이론) → B(구현) →
C(데이터셋 시각화) → D(시각화·해석) → E(결과분석) → F(트러블슈팅)입니다.

| 구분 | 내용 |
|---|---|
| A. 이론과 과제 정의 | 이 셀 — 물리 지배식과 결함 메커니즘 |
| B. 문제와 해결 및 구현 | 데이터 준비, 물리 모델·DL 모델·열화상 파이프라인·Explode 분석·가드레일·온라인 시뮬레이션 구현 |
| C. 데이터셋 시각화 | 원자료 자체의 분포(EDA) |
| D. 데이터 시각화 및 해석 | 각 모델의 피팅 결과 그래프와 해석 |
| E. 결과 정리 및 분석 | 전체 파이프라인을 통합한 최종 결론 |
| F. 트러블슈팅 및 개선 방안 | 이 세션에서 발견·수정한 모든 버그와 남은 한계 |

결론(E 섹션)만 먼저 읽어도 전체 파이프라인의 요지를 파악할 수 있도록 구성했지만, "왜 그 결론에
도달했는가"의 근거는 B~D 섹션의 물리·통계적 논증에 있으므로, 처음 읽으시는 분들께는 순서대로 읽기를
권합니다.


# B. 문제와 해결 및 구현

물리 모델(Bad 하한·너겟 성장·Explode 상한), 딥러닝 대리모델, 열화상 파이프라인, 가드레일, 온라인 학습
시뮬레이션까지 — 이 노트북에서 실제로 구현한 모든 파이프라인입니다. 각 구현 단계에서 발견한 버그와
수정 경위는 섹션 F(트러블슈팅)에 별도로 모아 정리했습니다.


## B-1 — 라이브러리 로드 및 전역 환경 설정

시드와 하이퍼파라미터를 이 셀에서 중앙 관리해 재현성을 보장합니다.


In [ ]:
# ==========================================
# [Cell 1] 라이브러리 로드 및 전역 환경 설정
# ==========================================
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from tqdm.notebook import tqdm

# 1. 폰트 및 시각화 설정 (Windows 환경 한글 깨짐 방지)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 그래프 가독성 상향: 축 눈금 숫자·축 라벨·제목·범례 텍스트를 전반적으로 키우고 굵게 고정
# (개별 셀에서 fontsize를 명시한 곳은 이 기본값을 덮어쓰므로 해당 값들도 별도로 상향했다)
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['legend.title_fontsize'] = 11
plt.rcParams['figure.titlesize'] = 16
plt.rcParams['figure.titleweight'] = 'bold'

# 2. 재현성을 위한 랜덤 시드 고정
RANDOM_SEED = 2026
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# 3. 데이터 경로 및 결과 저장 디렉토리
DATA_PATH = os.path.join(
    os.getcwd(), "Data", "Resistance Spot Welding Insights", "Data_RSW.csv"
)
IMG_DIR = os.path.join(os.getcwd(), "Data", "Resistance Spot Welding Insights", "ir_images")
RESULT_DIR = os.path.join("result_rsw")
FIGURE_DIR = os.path.join(RESULT_DIR, "figures")
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# 4. 연산 디바이스
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 5. 물리/통계 상수 및 하이퍼파라미터
NUM_BINS = 10                    # 분위수(qcut) 기반 구간 분할 수
MIN_BIN_SIZE = 15                # 구간별 최소 표본 수 (미달 시 병합/제외) — 지난 노트북의 표본 1개 구간 버그 재발 방지
TARGET_BAD_RATE = 0.05           # 허용 가능한 최대 미융착(Bad) 비율
FIT_MAX_FEV = 5000

BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
# 기존 100 epoch 고정 학습 시 val_loss가 epoch 9~15에서 최저(0.099)를 찍고 이후 계속 상승했음.
# CosineAnnealingLR의 T_max=NUM_EPOCHS로 연동돼 있어, epoch 9 시점 LR이 base LR의 98%로
# 거의 감쇠하지 않은 상태였던 게 원인 중 하나 — LR이 계속 높아 학습이 그 이후로도 train만
# 계속 파고들었다. NUM_EPOCHS를 20으로 줄이면 같은 cosine 스케줄이 epoch 15~18 부근에서
# 충분히 감쇠해 과적합 구간 진입을 막는지 확인한다 (T_max를 NUM_EPOCHS와 분리해 더 짧게
# 주면 cosine 공식이 주기적이라 T_max를 넘는 순간부터 LR이 다시 상승하므로 피해야 함).
NUM_EPOCHS = 20

print(f"[설정 완료] Seed: {RANDOM_SEED} | Device: {DEVICE}")


## B-1b — 물리 피처 엔지니어링 및 통계 분석 클래스 정의

**단계별 메서드:**

| 메서드 | 역할 |
|--------|------|
| `step1_load_and_aggregate()` | 샘플 ID별로 다중 행(통전 중 Force 시계열)을 집계, `Current=-99` 센서 오류 행 제외, 줄열 proxy $Q \propto I_{avg}^2 t$ 계산 |
| `step2_aggregate_distribution()` | `pd.qcut`으로 **표본 수가 균등한** 구간 분할 (등간격 `linspace` 대신 사용 — 표본 1개 구간 방지), 구간별 Bad 비율과 **Agresti-Coull 보정 SEM** 산출 (시각화용 요약 통계) |
| `step3_fit_defect_trend()` | 원본 샘플 전체(0/1 라벨)에 **감소형 로지스틱**을 직접 피팅, 목표 Bad 비율(5%) 이하를 만족하는 **최소 안전 열입력 $Q_{min,safe}$** 역산 |
| `step4_fit_nugget_growth()` | 구간 평균 ± SEM 대상으로 너겟 지름의 **포화 성장 곡선**(saturating exponential growth) 피팅 |

> **쉽게 말하면:** 표본이 몇 개 안 되는 구간에서는 Bad 비율이 우연히 0%나 100%로 나올 수 있는데, 이때 표준오차(SEM) 공식을 그대로 쓰면 "오차가 전혀 없다"는 비현실적인 결과가 나옵니다. Agresti-Coull 보정은 관측되지 않은 값이 있을 가능성을 고려해 이런 경우에도 합리적인 오차 크기를 계산해 줍니다.

> **SEM=0 버그 방지:** 이전 노트북은 표본 1개 구간의 SEM=0을 `curve_fit`의 `sigma`에서 1e-3으로 치환해 해당 구간이 사실상 무한대 가중치를 갖는 문제가 있었습니다.
> 여기서는 (1) `qcut`으로 애초에 구간별 표본 수를 균등하게 만들고, (2) 비율 데이터의 SEM은 Agresti-Coull 보정(`(x+2)/(n+4)` 기반)을 적용해 $p=0$ 또는 $1$이어도 SEM이 0이 되지 않도록 합니다.

> **피팅 발산 버그 방지 (Q_min 오차비율 87배, D_max=688mm 발산):** 열입력 구간이 10개뿐이고 그중 신호가 있는 구간이
> 1개(가장 낮은 구간)뿐이라 `step3`을 구간 평균으로 피팅하면 Q_min·k가 사실상 미결정 상태였습니다. `step3`은 구간화 이전
> 원본 493개 샘플 전체에 직접 피팅하도록 바꿔 자코비안을 잘 조건화시켰고, `step3`·`step4` 모두 물리적으로 타당한 범위로
> `bounds`를 지정해 파라미터가 비물리적인 값으로 발산하는 것을 차단했습니다.


In [ ]:
# ==========================================
# [Cell 2] 물리 피처 엔지니어링 및 통계 분석 클래스
# ==========================================
class ResistanceWeldingOptimizer:
    def __init__(self, data_path: str, num_bins: int, min_bin_size: int):
        self.data_path = data_path
        self.num_bins = num_bins
        self.min_bin_size = min_bin_size
        self.df = None
        self.binned_df = None

    @staticmethod
    def bad_rate_logistic_model(Q, Q_min, k):
        """열입력이 Q_min 아래로 떨어질수록 미융착(Bad) 확률이 증가하는 감소형 시그모이드"""
        return 1.0 / (1.0 + np.exp(k * (Q - Q_min)))

    @staticmethod
    def nugget_growth_model(Q, D0, D_max, tau):
        """열입력에 따라 D0에서 D_max로 포화 성장하는 너겟 지름 모델"""
        return D0 + (D_max - D0) * (1.0 - np.exp(-Q / tau))

    def step1_load_and_aggregate(self) -> pd.DataFrame:
        """샘플 단위 집계 및 줄열 proxy 계산"""
        raw = pd.read_csv(self.data_path, encoding="utf-8-sig")

        records = []
        for sample_id, grp in raw.groupby("Sample ID"):
            valid_current = grp.loc[grp["Current (A)"] > 0, "Current (A)"]
            if valid_current.empty:
                continue  # 유효 전류 계측치가 전혀 없는 샘플 제외 (-99 센서 오류만 존재)

            first = grp.iloc[0]
            weld_time_s = first["Welding Time (ms)"] / 1000.0
            avg_current = valid_current.mean()
            heat_input_proxy = (avg_current ** 2) * weld_time_s

            records.append({
                "sample_id": sample_id,
                "avg_current_A": avg_current,
                "peak_current_A": valid_current.max(),
                "weld_time_s": weld_time_s,
                "pressure_psi": first["Pressure (PSI)"],
                "angle_deg": first["Angle (Deg)"],
                "thickness_avg_mm": (first["Thickness A (mm)"] + first["Thickness B (mm)"]) / 2.0,
                "pull_test_N": first["PullTest (N)"],
                "nugget_diameter_mm": first["NuggetDiameter (mm)"],
                "category": first["Category"],
                "heat_input_proxy": heat_input_proxy,
                "is_bad": int(first["Category"] == "Bad"),
                "is_explode": int(first["Category"] == "Explode"),
            })

        self.df = pd.DataFrame(records)
        return self.df

    def step2_aggregate_distribution(self) -> pd.DataFrame:
        """열입력 분위수(qcut) 구간화 및 Bad 비율 / 너겟 지름 통계 산출 (시각화용 요약 통계)"""
        df = self.df.copy()
        df["heat_bin"] = pd.qcut(df["heat_input_proxy"], q=self.num_bins, duplicates="drop")

        rows = []
        for bin_label, grp in df.groupby("heat_bin", observed=True):
            n = len(grp)
            if n < self.min_bin_size:
                continue  # 표본 부족 구간 제외 (이전 버그의 근본 원인 재발 방지)

            successes = grp["is_bad"].sum()
            # Agresti-Coull 보정: p=0 또는 1이어도 SEM이 0이 되지 않음
            n_adj = n + 4
            p_adj = (successes + 2) / n_adj
            bad_sem = np.sqrt(p_adj * (1 - p_adj) / n_adj)

            rows.append({
                "heat_mean": grp["heat_input_proxy"].mean(),
                "n_samples": n,
                "bad_rate": grp["is_bad"].mean(),
                "bad_rate_sem": bad_sem,
                "nugget_mean": grp["nugget_diameter_mm"].mean(),
                "nugget_sem": grp["nugget_diameter_mm"].std(ddof=1) / np.sqrt(n),
            })

        self.binned_df = pd.DataFrame(rows).sort_values("heat_mean").reset_index(drop=True)
        return self.binned_df

    def step3_fit_defect_trend(self) -> pd.DataFrame:
        """미융착(Bad) 확률 로지스틱 피팅 및 최소 안전 열입력 도출

        구간화된 10개 평균점만으로 피팅하면 표본 1개 구간(가장 낮은 열입력 구간, bad_rate=0.36)에만
        실질적인 신호가 있고 나머지 9개 구간은 거의 0이라, Q_min/k가 사실상 미결정 상태(오차비율 87배)로
        발산했습니다. 여기서는 구간화 이전의 원본 샘플 전체(0/1 라벨)에 직접 곡선을 피팅해 자코비안을
        훨씬 잘 조건화(well-conditioned)시키고, bounds로 비물리적인 해를 차단합니다.
        """
        x = self.df["heat_input_proxy"].values
        y = self.df["is_bad"].values.astype(float)
        x_range = x.max() - x.min()

        # 초기값: 열입력 하위 15% 지점을 문턱 근처로, 가파른 전이를 가정한 초기 기울기
        q_min0 = np.percentile(x, 15)
        k0 = 10.0 / x_range

        popt, pcov = curve_fit(
            self.bad_rate_logistic_model, x, y,
            p0=[q_min0, k0],
            bounds=([x.min(), 1e-8], [x.max(), 50.0 / x_range]),
            maxfev=20000,
        )
        q_min_opt, k_opt = popt
        q_min_err = np.sqrt(np.diag(pcov))[0]

        # Bad 비율이 목표치(5%) 이하가 되는 최소 안전 열입력
        safe_q_min = q_min_opt + np.log((1.0 / TARGET_BAD_RATE) - 1.0) / k_opt
        fit_reliability = q_min_err / q_min_opt  # 값 대비 오차 비율 — 피팅 신뢰도 지표

        return pd.DataFrame([{
            "q_min_opt": q_min_opt,
            "q_min_err": q_min_err,
            "slope_k_opt": k_opt,
            "safe_heat_min": safe_q_min,
            "target_bad_rate": TARGET_BAD_RATE,
            "fit_reliability_ratio": fit_reliability,
        }])

    def step4_fit_nugget_growth(self) -> pd.DataFrame:
        """너겟 지름 포화 성장 곡선 피팅 (구간 평균 ± SEM 대상, bounds로 비물리적 해 차단)

        bounds 없이 피팅하면 관측 구간 내 성장이 거의 선형이라 D_max·tau가 개별적으로 미결정되어
        (tau=2.3e10, D_max=688mm 같은) 비물리적인 해로 발산했습니다. D_max는 관측 최댓값의 3배,
        tau는 관측 열입력 범위의 0.05~20배로 제한해 물리적으로 타당한 해만 남깁니다.
        """
        x = self.binned_df["heat_mean"].values
        y = self.binned_df["nugget_mean"].values
        yerr = self.binned_df["nugget_sem"].values
        x_range = x.max() - x.min()

        popt, pcov = curve_fit(
            self.nugget_growth_model, x, y,
            p0=[y.min() * 0.95, y.max() * 1.1, x_range],
            sigma=yerr, absolute_sigma=True, maxfev=FIT_MAX_FEV,
            bounds=([0.0, y.max(), x_range * 0.05], [y.min(), y.max() * 3.0, x_range * 20.0]),
        )
        d0_opt, dmax_opt, tau_opt = popt
        perr = np.sqrt(np.diag(pcov))

        return pd.DataFrame([{
            "D0_opt": d0_opt, "Dmax_opt": dmax_opt, "tau_opt": tau_opt,
            "D0_err": perr[0], "Dmax_err": perr[1], "tau_err": perr[2],
        }])


## B-1c — 파이프라인 실행 및 중간 결과 CSV 저장

**출력 파일 (`result_rsw/` 폴더):**

| 파일명 | 내용 |
|--------|------|
| `step1_aggregated_samples.csv` | 샘플 단위 집계 데이터 (495건 → 유효 전류 보유 샘플만) |
| `step2_binned_stats.csv` | 열입력 구간별 Bad 비율 / 너겟 지름 평균·SEM |
| `step3_defect_fit_params.csv` | Bad 확률 로지스틱 피팅 파라미터 및 신뢰도 지표 |
| `step4_nugget_fit_params.csv` | 너겟 성장 곡선 피팅 파라미터 |


In [ ]:
# ==========================================
# [Cell 3] 클래스 인스턴스화 및 단계별 데이터 저장
# ==========================================
optimizer_engine = ResistanceWeldingOptimizer(
    data_path=DATA_PATH, num_bins=NUM_BINS, min_bin_size=MIN_BIN_SIZE
)

df_agg = optimizer_engine.step1_load_and_aggregate()
df_agg.to_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"), index=False)

df_binned = optimizer_engine.step2_aggregate_distribution()
df_binned.to_csv(os.path.join(RESULT_DIR, "step2_binned_stats.csv"), index=False)

df_defect_fit = optimizer_engine.step3_fit_defect_trend()
df_defect_fit.to_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv"), index=False)

df_nugget_fit = optimizer_engine.step4_fit_nugget_growth()
df_nugget_fit.to_csv(os.path.join(RESULT_DIR, "step4_nugget_fit_params.csv"), index=False)

print(f"[완료] 유효 샘플 {len(df_agg)}건 / 구간 {len(df_binned)}개 (min_bin_size={MIN_BIN_SIZE} 적용)")
print(f"[완료] 단계별 CSV 파일이 {RESULT_DIR}/ 폴더에 저장되었습니다.")
display(df_defect_fit)
display(df_nugget_fit)


## B-2 — 딥러닝 대리 모델 (다변량 결함 확률 예측)

**입력 피처:** `avg_current_A`, `weld_time_s`, `pressure_psi`, `thickness_avg_mm`
**출력:** Bad 여부(0/1) 확률
**구조:** `입력(4) → Linear(32) → BatchNorm1d → SiLU → Linear(16) → SiLU → Linear(1) → Sigmoid`


In [ ]:
# ==========================================
# [Cell 6] 대리 모델 정의 및 학습
# ==========================================
df_model = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))
feature_cols = ['avg_current_A', 'weld_time_s', 'pressure_psi', 'thickness_avg_mm']
X_raw = df_model[feature_cols].values
y_raw = df_model['is_bad'].values

X_min, X_max = X_raw.min(axis=0), X_raw.max(axis=0)
pd.DataFrame({'feature': feature_cols, 'min_val': X_min, 'max_val': X_max}) \
    .to_csv(os.path.join(RESULT_DIR, "scaler_params.csv"), index=False)
X_norm = (X_raw - X_min) / (X_max - X_min + 1e-8)

dataset = TensorDataset(torch.tensor(X_norm, dtype=torch.float32),
                         torch.tensor(y_raw, dtype=torch.float32).unsqueeze(1))
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(RANDOM_SEED))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

class RSWDefectNet(nn.Module):
    def __init__(self, input_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), nn.BatchNorm1d(32), nn.SiLU(),
            nn.Linear(32, 16), nn.SiLU(),
            nn.Linear(16, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)

model = RSWDefectNet(input_dim=len(feature_cols)).to(DEVICE)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

history = []
pbar = tqdm(range(1, NUM_EPOCHS + 1), desc="학습")
for epoch in pbar:
    current_lr = optimizer.param_groups[0]['lr']
    model.train()
    train_loss_total = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(DEVICE), by.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * bx.size(0)

    model.eval()
    val_loss_total, val_preds, val_targets = 0.0, [], []
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            preds = model(bx)
            val_loss_total += criterion(preds, by).item() * bx.size(0)
            val_preds.extend((preds >= 0.5).float().cpu().numpy().flatten())
            val_targets.extend(by.cpu().numpy().flatten())

    train_loss = train_loss_total / train_size
    val_loss = val_loss_total / val_size
    val_acc = np.mean(np.array(val_preds) == np.array(val_targets)) * 100.0
    val_f1 = f1_score(val_targets, val_preds, zero_division=0)

    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                     'val_accuracy': val_acc, 'val_f1_score': val_f1, 'lr': current_lr})
    scheduler.step()
    pbar.set_postfix(TL=f"{train_loss:.4f}", VL=f"{val_loss:.4f}",
                      F1=f"{val_f1:.3f}", Acc=f"{val_acc:.1f}%")

pd.DataFrame(history).to_csv(os.path.join(RESULT_DIR, "step5_training_history.csv"), index=False)
torch.save(model.state_dict(), os.path.join(RESULT_DIR, "rsw_defect_model.pth"))
print(f"[완료] 학습 종료 — 최종 Val F1={history[-1]['val_f1_score']:.3f}, "
      f"Val Acc={history[-1]['val_accuracy']:.1f}%")


## B-3 — 실시간 파라미터 가드레일 시뮬레이션

AI가 제안한 용접 조건(전류/시간)의 열입력이 안전 하한에 못 미치면(미융착 위험) 통전 시간을 자동 보정합니다.
Explode(과열) 상한까지 포함한 완전한 버전은 B-6에서 다룹니다.


In [ ]:
# ==========================================
# [Cell 8] 열입력 부족 차단 가드레일
# ==========================================
class HeatDeficiencyGuardrail:
    """제안된 전류/시간의 열입력이 최소 안전 기준 미달일 경우 통전 시간을 자동 보정"""
    def __init__(self, safe_heat_min: float):
        self.safe_heat_min = safe_heat_min

    def validate_and_control(self, current: float, weld_time_s: float) -> dict:
        calc_heat = (current ** 2) * weld_time_s
        if calc_heat < self.safe_heat_min:
            corrected_time = self.safe_heat_min / (current ** 2)
            return {
                "guardrail_status": "TRIPPED_UNDERHEAT",
                "input_heat": calc_heat,
                "corrected_weld_time_s": corrected_time,
                "action": f"미융착 위험: 통전시간 보정 ({weld_time_s:.3f}s -> {corrected_time:.3f}s)",
            }
        return {
            "guardrail_status": "NORMAL_PASS",
            "input_heat": calc_heat,
            "corrected_weld_time_s": weld_time_s,
            "action": "정상 범위 내 통과",
        }

fit = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]
guardrail = HeatDeficiencyGuardrail(safe_heat_min=fit['safe_heat_min'])

test_scenarios = [
    {"scenario": "정상 공정", "current": 2000.0, "weld_time_s": 0.8},
    {"scenario": "비정상 저열입력(환각)", "current": 900.0, "weld_time_s": 0.2},
]
log_results = [{**s, **guardrail.validate_and_control(s['current'], s['weld_time_s'])} for s in test_scenarios]
df_log = pd.DataFrame(log_results)
df_log.to_csv(os.path.join(RESULT_DIR, "step6_guardrail_log.csv"), index=False)
display(df_log[['scenario', 'guardrail_status', 'input_heat', 'corrected_weld_time_s', 'action']])


## B-4 — 열화상 이미지 기반 물리적 특징 추출 (머신비전 확장)

**데이터 제약 (중요):** IR 이미지는 전체 493개 유효 샘플 중 **99개(Sample ID 10~18, 100~189 구간)** 에만 존재하고,
결함 라벨 분포가 `Good 92 / Bad 6 / Explode 1`로 극심하게 불균형합니다. Sample ID가 특정 구간에 몰려 있는 것으로 보아
전체 실험을 무작위로 대표하는 표본이 아니라 **특정 실험 세션에서만 촬영된 부분집합**입니다. 이 때문에 CNN 다중분류
학습(Explode 1장으로는 학습/검증 분리조차 불가능)은 시도하지 않고, 대신 **이미지 픽셀에서 물리적으로 해석 가능한
특징을 추출**해 기존 열입력(Q) · 너겟 지름 피팅과 상관분석하는 접근을 취합니다.

**이미지를 직접 열어본 뒤 발견한 두 가지 결정적 문제:**

1. **Jet류 컬러맵**이라 밝기(luminance)가 온도와 단조적으로 대응하지 않습니다. 배경(저온)은 파랑, 중간은
   초록·노랑, 고온부는 빨강/주황인데, ITU-R BT.601 밝기 가중치(`0.299R+0.587G+0.114B`)는 초록·노랑을
   빨강보다 훨씬 밝게 취급합니다.
2. **모든 이미지에 흰색 오버레이가 찍혀 있습니다**: 용접부 위치를 가리키는 흰색 십자선(crosshair) 마커와
   화면 하단의 흰색 `"10mm"` 축척바(눈금선 + 텍스트). 흰색은 밝기가 항상 255에 가까워, 밝기 기반 버전에서는
   **실제 열 신호보다 이 오버레이가 "가장 뜨거운 영역"으로 오검출**됐습니다. 특히 열입력이 낮아 실제 열 신호가
   약한 샘플일수록 오버레이가 상대적으로 더 두드러져, **하필 결함 여부를 가르는 저열입력 구간에서 오염이
   가장 심했습니다** (Sample 188·17·16에서 확인).

**2단계 전처리로 대응합니다:**

1. **축척바 크롭**: 화면 하단의 흰색 눈금선(다소 기울어져 있어도 허용)을 연결영역 분석으로 찾아, 그 선의
   **최상단 행(가장 기울어져 튀어나온 지점 기준)에서 선 두께만큼 더 위로 올라간 행을 기준으로 그 아래 전체를
   잘라냅니다.** 선의 두께는 각 열(column)에서 흰색이 위에서부터 연속되는 픽셀 수의 중앙값으로 추정해,
   "10mm" 텍스트가 선과 붙어 있어도 국소적으로 재는 방식이라 안정적입니다. 축척바를 못 찾으면 크롭 없이 원본을
   그대로 사용합니다(안전한 폴백).
2. **채도·밝기 마스킹**: 크롭된 이미지에서 RGB→HSV 변환 후 채도가 낮은 픽셀(흰색 잔여 오버레이)과 밝기가
   비정상적으로 낮은 픽셀(검은색 잔여 오버레이)을 함께 제외하고, 남은 유효 픽셀의 Hue(색상각)를
   0(빨강=고온)~1(파랑=저온) `warmth` 점수로 변환합니다.

   > **십자선 마커의 검은색 테두리 누락 방지:** `IR_14.jpg`를 보면 십자선 마커가 흰색 선 + 검은색 테두리의
   > 이중선으로 그려져 있습니다. 흰색은 채도 필터로 걸러지지만, 검은색 테두리는 JPEG 압축의 색 번짐으로
   > 채도가 살짝 남아 필터를 통과할 위험이 있습니다. Jet 컬러맵은 가장 차가운 파랑도 밝기가 어느 정도
   > 있어(완전한 검정까지 가지 않음) 밝기가 비정상적으로 낮은 픽셀은 배경이 아니라 오버레이일 가능성이
   > 높다는 점을 이용해, 채도 하한과 대칭으로 **밝기 하한(`MIN_VALID_VALUE=0.2`)** 조건을 추가했습니다.
   > 이미지를 통째로 배제하는 대신 픽셀 단위로만 제외해 표본 수(99장)를 그대로 유지합니다.

**추출 특징:**

| 특징 | 물리적 의미 |
|---|---|
| `peak_warmth` | 유효 픽셀 중 최대 warmth — 피크 온도 proxy |
| `mean_warmth` | 유효 픽셀의 평균 warmth — 열 확산 정도 proxy |
| `hot_area_fraction` | 전역 임계값(유효 픽셀 warmth 분포의 상위 15%) 이상인 유효 픽셀의 비율 — 용융풀/열영향부 크기 proxy |
| `hotspot_centroid_offset_px` | 최대 연결 고온 영역의 무게중심이 이미지 중심에서 벗어난 거리 — 용융풀 비대칭(처짐) proxy |
| `hotspot_aspect_ratio` | 최대 연결 고온 영역의 bounding box 종횡비 — 용융풀 형상(신장도) proxy |

> **QA:** 이미지별 크롭 행 수를 `step7b_scalebar_crop_debug.csv`에 저장합니다. 축척바를 못 찾은 이미지가
> 있으면(폴백 발생) 여기서 바로 확인할 수 있습니다.

**OpenCV 적용:** HSV 변환(`matplotlib.colors.rgb_to_hsv` → `cv2.cvtColor`)과 연결영역 분석
(`scipy.ndimage.label` → `cv2.connectedComponentsWithStats`)을 표준 OpenCV API로 교체했습니다.
`connectedComponentsWithStats`는 라벨링과 동시에 bbox·area·centroid를 한 번에 반환해 코드도 더
간결해졌습니다. 축척선 검출은 `cv2.HoughLinesP`로 독립 교차검증하는 QA 셀을 별도로 추가했습니다.


In [ ]:
# ==========================================
# [Cell 10] 열화상 이미지 물리 특징 추출 (축척바 크롭 + Hue 기반 warmth + 채도 마스킹)
# ==========================================
import cv2  # pip install opencv-python

def to_hsv(img_rgb: np.ndarray):
    """cv2.cvtColor는 8비트 입력 기준 H:0~179, S/V:0~255 범위를 쓰므로,
    기존 임계값 상수(0~1, 0~360 기준)를 그대로 재사용할 수 있게 정규화해 반환한다."""
    hsv_u8 = cv2.cvtColor(img_rgb[..., :3].astype(np.uint8), cv2.COLOR_RGB2HSV)
    hue_deg = hsv_u8[..., 0].astype(np.float64) * (360.0 / 179.0)
    sat = hsv_u8[..., 1].astype(np.float64) / 255.0
    val = hsv_u8[..., 2].astype(np.float64) / 255.0
    return hue_deg, sat, val

SAT_THRESHOLD = 0.15        # 이 채도 미만은 흰색/회색 오버레이로 간주해 제외
VAL_THRESHOLD = 0.8         # 축척바 검출용 "흰색" 판정 밝기(Value) 하한
MIN_VALID_VALUE = 0.2       # 이 밝기 미만(검정에 가까움)도 오버레이(십자선 테두리 등)로 간주해 제외
MIN_LINE_WIDTH_FRAC = 0.3   # 이미지 너비의 이 비율 이상 길어야 "축척 눈금선" 후보로 인정
MAX_LINE_THICKNESS_FRAC = 0.15  # 대표 두께가 폭 대비 이보다 두꺼우면 선이 아닌 다른 도형으로 판단
HOT_PERCENTILE = 85
MIN_BLOB_PIXELS = 20        # 노이즈성 소규모 영역을 걸러내는 보조 안전장치

def find_scalebar_crop_row(img_rgb: np.ndarray):
    """화면 하단 축척바의 흰색 눈금선을 찾아 (선의 최상단 행, 선 두께)를 반환한다.
    선이 다소 기울어져 있어도 연결영역 전체의 최상단 행을 그대로 사용해 대응하고,
    두께는 각 열(column)에서 흰색이 위에서부터 연속되는 픽셀 수의 **중앙값**으로 국소 추정한다.
    "10mm" 텍스트가 선 바로 아래 붙어 일부 열에서만 두꺼워지더라도, 전체 폭에 걸친 열들의
    중앙값은 대부분 순수 선 두께에 가깝게 유지되므로 bbox 전체 높이보다 안정적이다."""
    h, w = img_rgb.shape[:2]
    _, sat, val = to_hsv(img_rgb)
    white_mask = (sat < SAT_THRESHOLD) & (val > VAL_THRESHOLD)

    n_labels, labeled = cv2.connectedComponents(white_mask.astype(np.uint8), connectivity=8)
    best = None  # (bbox_width, top_row, thickness)
    for label_id in range(1, n_labels):
        ys, xs = np.nonzero(labeled == label_id)
        bbox_w = xs.max() - xs.min() + 1
        if bbox_w < MIN_LINE_WIDTH_FRAC * w:
            continue  # 너무 짧으면(십자선·낱글자) 축척선 후보에서 제외

        top_per_col, thickness_per_col = [], []
        for x in range(xs.min(), xs.max() + 1):
            col_ys = ys[xs == x]
            if col_ys.size == 0:
                continue
            y0 = int(col_ys.min())
            top_per_col.append(y0)
            run = 1
            while y0 + run < h and white_mask[y0 + run, x]:
                run += 1
            thickness_per_col.append(run)

        if not top_per_col:
            continue
        thickness = int(np.median(thickness_per_col))
        if thickness > MAX_LINE_THICKNESS_FRAC * bbox_w:
            continue  # 대표 두께가 폭 대비 너무 크면 선이 아니라 글자 뭉치 등으로 판단

        top_row = int(np.min(top_per_col))  # 기울어진 선 전체의 최상단
        if best is None or bbox_w > best[0]:
            best = (bbox_w, top_row, thickness)

    return (best[1], best[2]) if best else None

def crop_above_scalebar(img_rgb: np.ndarray):
    """축척바를 찾으면 (최상단 행 - 선 두께) 위쪽만 남기고 자른다. 못 찾으면 원본 그대로 반환(폴백)."""
    found = find_scalebar_crop_row(img_rgb)
    if found is None:
        return img_rgb, None
    top_row, thickness = found
    cutoff = max(top_row - thickness, 1)
    return img_rgb[:cutoff, :, :], cutoff

def compute_warmth(img_rgb: np.ndarray):
    """Jet류 컬러맵(빨강=고온 ~ 파랑=저온)을 Hue로 복원해 상대적 열 강도로 변환하고,
    오버레이 픽셀(십자선 마커 등)은 유효 픽셀에서 제외한다.

    십자선 마커는 흰색 선 + 검은색 테두리의 이중선으로 그려져 있다 (IR_14.jpg에서 확인).
    흰색 부분은 채도가 낮아 기존 필터로 걸러지지만, 검은색 테두리는 JPEG 압축 시 색 번짐으로
    채도가 남아 필터를 통과할 수 있다. Jet 컬러맵은 가장 차가운 파랑도 밝기가 어느 정도 있어
    (완전한 검정까지 가지 않음), 밝기가 비정상적으로 낮은 픽셀은 배경이 아니라 오버레이일
    가능성이 높다 — 채도 기준과 대칭으로 밝기 하한 조건을 추가해 흰색·검은색 오버레이를 함께 제외한다."""
    hue_deg, sat, val = to_hsv(img_rgb)
    valid = (sat > SAT_THRESHOLD) & (val > MIN_VALID_VALUE)
    warmth = 1.0 - np.clip(hue_deg / 240.0, 0.0, 1.0)  # 0(파랑/저온) ~ 1(빨강/고온)
    return warmth, valid

def extract_thermal_features(warmth: np.ndarray, valid: np.ndarray, threshold: float,
                              min_blob_pixels: int = MIN_BLOB_PIXELS) -> dict:
    """단일 IR 이미지에서 물리적으로 해석 가능한 열 특징을 추출 (오버레이 픽셀 제외)."""
    h, w = warmth.shape
    cy0, cx0 = h / 2.0, w / 2.0
    mask = valid & (warmth > threshold)

    valid_warmth = warmth[valid]
    feat = {
        "peak_warmth": float(valid_warmth.max()) if valid_warmth.size else np.nan,
        "mean_warmth": float(valid_warmth.mean()) if valid_warmth.size else np.nan,
        "hot_area_fraction": float(mask.sum()) / float(valid.sum()) if valid.sum() else np.nan,
        "hotspot_centroid_offset_px": np.nan,
        "hotspot_aspect_ratio": np.nan,
    }
    if mask.sum() == 0:
        return feat

    # cv2.connectedComponentsWithStats: 라벨링과 동시에 bbox(x,y,w,h)·area·centroid를 한 번에 반환
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8)
    if n_labels <= 1:  # 배경(0)만 있고 전경 라벨이 없음
        return feat

    fg_areas = stats[1:, cv2.CC_STAT_AREA]  # 라벨 0=배경 제외
    largest_label = 1 + int(np.argmax(fg_areas))
    blob_size = int(fg_areas.max())
    if blob_size < min_blob_pixels:
        return feat

    cx, cy = centroids[largest_label]
    feat["hotspot_centroid_offset_px"] = float(np.hypot(cy - cy0, cx - cx0))

    bbox_w = int(stats[largest_label, cv2.CC_STAT_WIDTH])
    bbox_h = int(stats[largest_label, cv2.CC_STAT_HEIGHT])
    feat["hotspot_aspect_ratio"] = max(bbox_h, bbox_w) / max(min(bbox_h, bbox_w), 1)
    return feat

ir_files = sorted(f for f in os.listdir(IMG_DIR) if f.lower().endswith(".jpg"))
sample_ids_with_ir = [int(f.split("_")[1].split(".")[0]) for f in ir_files]

warmth_valid = {}
crop_log = []
for sid, fname in zip(sample_ids_with_ir, ir_files):
    img = plt.imread(os.path.join(IMG_DIR, fname))
    cropped_img, cutoff = crop_above_scalebar(img)
    crop_log.append({
        "sample_id": sid, "orig_height": img.shape[0],
        "cropped_height": cropped_img.shape[0], "scalebar_found": cutoff is not None,
    })
    warmth_valid[sid] = compute_warmth(cropped_img)

crop_log_df = pd.DataFrame(crop_log)
crop_log_df.to_csv(os.path.join(RESULT_DIR, "step7b_scalebar_crop_debug.csv"), index=False)
n_not_found = (~crop_log_df["scalebar_found"]).sum()
print(f"[축척바 크롭] {len(crop_log_df)}장 중 {n_not_found}장은 축척선을 못 찾아 원본 그대로 사용(폴백)")
print(f"[축척바 크롭] 잘린 높이(원본 대비) 중앙값: "
      f"{(crop_log_df['orig_height'] - crop_log_df['cropped_height']).median():.0f}px")

# 전체 이미지의 유효(고채도) 픽셀만 pooling한 전역 임계값
all_valid_warmth = np.concatenate([w[v] for w, v in warmth_valid.values()])
HOT_THRESHOLD = np.percentile(all_valid_warmth, HOT_PERCENTILE)
print(f"[전역 임계값] 유효(고채도) 픽셀 상위 {100 - HOT_PERCENTILE}% 기준 HOT_THRESHOLD(warmth) = {HOT_THRESHOLD:.3f}")

thermal_records = []
for sid, (warmth, valid) in warmth_valid.items():
    feat = extract_thermal_features(warmth, valid, HOT_THRESHOLD)
    feat["sample_id"] = sid
    thermal_records.append(feat)

thermal_df = pd.DataFrame(thermal_records)
n_excluded = thermal_df["hotspot_aspect_ratio"].isna().sum()
print(f"[블롭 크기 가드] 최대 연결영역 < {MIN_BLOB_PIXELS}px 인 이미지 {n_excluded}장은 형상 특징을 NaN 처리")

thermal_df.to_csv(os.path.join(RESULT_DIR, "step7_thermal_features.csv"), index=False)
thermal_df.describe().to_csv(os.path.join(RESULT_DIR, "step7c_thermal_features_describe.csv"))
print(f"[완료] IR 이미지 {len(thermal_df)}장에서 물리 특징(축척바 크롭 + Hue 기반 warmth) 재추출")
thermal_df.describe()


## B-4b — OpenCV `HoughLinesP`로 축척바 검출 교차검증

기존 축척선 검출(`find_scalebar_crop_row`)은 열(column) 단위 스캔으로 직접 구현한 방식입니다.
여기서는 OpenCV의 표준 직선 검출 함수 `cv2.HoughLinesP`로 같은 흰색 마스크에서 독립적으로 선을
찾아, 두 방법이 검출한 최상단 행이 서로 5px 이내로 일치하는지 교차검증합니다. `HoughLinesP`는
선분의 두께까지는 알려주지 않아 크롭 기준값 계산(두께 추정)에는 여전히 기존 방식을 쓰지만,
"정말 선을 찾은 게 맞는지"를 독립적인 알고리즘으로 재확인하는 용도로 유용합니다.

**출력 파일:** `result_rsw/step7d_hough_crosscheck.csv`


In [ ]:
# ==========================================
# [B-4b] OpenCV HoughLinesP 교차검증
# ==========================================
hough_log = []
for sid, fname in zip(sample_ids_with_ir, ir_files):
    img = plt.imread(os.path.join(IMG_DIR, fname))
    _, sat, val = to_hsv(img)
    white_mask_u8 = ((sat < SAT_THRESHOLD) & (val > VAL_THRESHOLD)).astype(np.uint8) * 255
    min_len = int(MIN_LINE_WIDTH_FRAC * img.shape[1])

    lines = cv2.HoughLinesP(white_mask_u8, rho=1, theta=np.pi / 180, threshold=min_len,
                             minLineLength=min_len, maxLineGap=10)
    if lines is not None:
        ys = np.concatenate([lines[:, 0, 1], lines[:, 0, 3]])
        hough_row = int(ys.min())
    else:
        hough_row = None

    custom_result = find_scalebar_crop_row(img)
    custom_row = custom_result[0] if custom_result else None

    hough_log.append({
        "sample_id": sid, "hough_top_row": hough_row, "custom_top_row": custom_row,
        "agree_within_5px": bool(hough_row is not None and custom_row is not None
                                  and abs(hough_row - custom_row) <= 5),
    })

hough_df = pd.DataFrame(hough_log)
hough_df.to_csv(os.path.join(RESULT_DIR, "step7d_hough_crosscheck.csv"), index=False)

n_both = hough_df.dropna(subset=["hough_top_row", "custom_top_row"]).shape[0]
n_agree = hough_df["agree_within_5px"].sum()
n_hough_missed = hough_df["hough_top_row"].isna().sum()
print(f"[HoughLinesP 교차검증] 기존 방식과 Hough 둘 다 검출된 {n_both}장 중 {n_agree}장이 5px 이내로 일치")
print(f"[참고] Hough가 아예 못 찾은 이미지: {n_hough_missed}장 (기존 열-스캔 방식이 더 안정적임을 시사)")
hough_df.head(10)


## B-5 — Explode(팽출) 결함 요인 탐색 및 상한 모델 피팅

샘플 단위 집계에서 `Explode`는 31건(6.3%)으로 `Bad`(21건, 4.3%)보다 오히려 흔한 결함인데도 지금까지 정량 분석
대상에서 제외돼 있었습니다 (서두 마크다운의 "열입력과 무관"이라는 서술도 코드로 검증된 적 없는 주장이었습니다).

1. 후보 변수(열입력 Q, 가압력, 전극각도, 피크전류, 판재두께) 각각과 `is_explode`의 점이연 상관(point-biserial
   correlation)을 계산해 실제로 유의미한 단변량 요인이 있는지 확인합니다.
2. p<0.05인 변수가 있으면, 그 변수의 **고유값 개수에 따라 분석 방식을 분기**합니다.
   - **고유값 ≤10 (범주형, 예: `angle_deg`의 0°/15°)**: 연속 로지스틱을 억지로 피팅하지 않고 **수준별
     실측 Explode 비율**을 직접 비교합니다.
   - **고유값 >10 (연속형)**: D-1의 결함 하한 피팅과 동일한 전략(bounds 포함 원본 샘플 피팅)으로
     아래의 **증가형 로지스틱**을 피팅해 안전 상한을 구합니다.

     $$p(X) = \frac{1}{1 + e^{-k(X - X_{mid})}}$$

     D-1의 감소형 로지스틱과 지수 항의 부호만 반대입니다 — Explode는 "값이 클 때" 증가하는 결함이므로
     $X \to \infty$ 이면 $p \to 1$, $X \to -\infty$ 이면 $p \to 0$인 증가형을 씁니다. $X_{mid}$는
     $p(X_{mid})=0.5$가 되는 문턱값입니다.
3. 유의미한 변수가 없으면 억지로 모델을 만들지 않고 "단변량으로는 원인 불명"이라고 정직하게 보고합니다.

> **로지스틱 피팅이 근본적으로 잘못됐던 문제:** 처음엔 채택된 변수(`angle_deg`, 0°/15° 두 값뿐)에도 무조건
> 연속 로지스틱을 피팅했습니다. 두 x값만으로는 $X_{mid}$·$k$가 수학적으로 유일하게 결정되지 않아, 최적화기가
> 경계값($X_{mid}=15$)에 붙잡힌 채 종료됐고, 그 결과 `angle=15`의 실측 비율(9.8%)과 피팅 곡선의 예측값(50%)이
> 5배 넘게 어긋났습니다 — 그래프의 점(qcut 붕괴로 1개만 표시되던 것도 별개 버그로 함께 있었음)과 곡선이
> 서로 다른 이야기를 하고 있었던 셈입니다. 근본 원인이 "범주형 변수에 연속 모델을 강제한 것"이었으므로,
> 위 1)/2) 분기로 아예 다른 접근을 쓰도록 재설계했습니다.

**출력 파일:** `result_rsw/step10_explode_driver_exploration.csv`, `result_rsw/step11_explode_fit_params.csv`,
`result_rsw/figures/explode_process_window.png`


In [ ]:
# ==========================================
# [Cell 13] Explode 결함 요인 탐색 + 상한 모델 피팅
# ==========================================
from scipy.stats import pointbiserialr

agg_explode = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))
explode_dist_df = pd.DataFrame([{
    "n_explode": int(agg_explode['is_explode'].sum()), "n_total": len(agg_explode),
    "explode_rate": agg_explode['is_explode'].mean(),
}])
explode_dist_df.to_csv(os.path.join(RESULT_DIR, "step10b_explode_distribution.csv"), index=False)
print(f"[Explode 분포] {agg_explode['is_explode'].sum()}건 / 전체 {len(agg_explode)}건 "
      f"({agg_explode['is_explode'].mean()*100:.1f}%)")

candidates = ["heat_input_proxy", "pressure_psi", "angle_deg", "peak_current_A", "thickness_avg_mm"]
explore_rows = []
for col in candidates:
    r, p = pointbiserialr(agg_explode["is_explode"], agg_explode[col])
    explore_rows.append({"변수": col, "r": r, "p_value": p})

explore_df = pd.DataFrame(explore_rows)
explore_df = explore_df.reindex(explore_df["r"].abs().sort_values(ascending=False).index).reset_index(drop=True)
explore_df.to_csv(os.path.join(RESULT_DIR, "step10_explode_driver_exploration.csv"), index=False)
display(explore_df)

best = explore_df.iloc[0]
EXPLODE_DRIVER = None
if best["p_value"] < 0.05:
    EXPLODE_DRIVER = best["변수"]
    print(f"[채택] '{EXPLODE_DRIVER}' 이(가) Explode와 가장 강한 단변량 관계 "
          f"(r={best['r']:.3f}, p={best['p_value']:.4f})")
else:
    print(f"[결론] 후보 변수 중 p<0.05인 것이 없습니다 (최선: {best['변수']} r={best['r']:.3f}, "
          f"p={best['p_value']:.4f}). 단변량으로는 Explode의 명확한 원인을 특정할 수 없어, "
          "상한 모델 피팅은 생략합니다.")

TARGET_EXPLODE_RATE = 0.05

if EXPLODE_DRIVER is not None:
    x = agg_explode[EXPLODE_DRIVER].values.astype(float)
    y = agg_explode["is_explode"].values.astype(float)
    n_unique = pd.Series(x).nunique()
    EXPLODE_MODE = "categorical" if n_unique <= 10 else "continuous"

    if EXPLODE_MODE == "categorical":
        # angle_deg처럼 두어 개 수준뿐인 변수에 연속 로지스틱을 최소자승으로 피팅하면
        # X_mid·k가 수학적으로 유일하게 결정되지 않는다. 실제로 시도해보니 최적화기가
        # 경계값(X_mid=x.max())에 붙잡혀, 그 수준에서의 실측 비율(9.8%)과 피팅 곡선의
        # 예측값(50%)이 5배 넘게 어긋나는 문제가 있었다. 따라서 범주형 변수는 로지스틱을
        # 강제하지 않고 수준별 실측 비율을 직접 비교한다.
        level_stats = pd.DataFrame({EXPLODE_DRIVER: x, "is_explode": y}) \
            .groupby(EXPLODE_DRIVER, observed=True) \
            .agg(n=("is_explode", "size"), rate=("is_explode", "mean")) \
            .reset_index()
        n_adj = level_stats["n"] + 4
        p_adj = (level_stats["rate"] * level_stats["n"] + 2) / n_adj
        level_stats["sem"] = np.sqrt(p_adj * (1 - p_adj) / n_adj)
        level_stats["safe"] = level_stats["rate"] < TARGET_EXPLODE_RATE
        safe_levels = level_stats.loc[level_stats["safe"], EXPLODE_DRIVER].tolist()

        print(f"[Explode 범주형 요인] 기준 변수: {EXPLODE_DRIVER} (고유값 {n_unique}개 — 수준별 비교)")
        for _, row in level_stats.iterrows():
            tag = "안전" if row["safe"] else "위험 증가"
            print(f"  {EXPLODE_DRIVER}={row[EXPLODE_DRIVER]:g}: n={int(row['n'])}, "
                  f"Explode 비율={row['rate']*100:.1f}% ± {row['sem']*100:.1f}%p  [{tag}]")

        explode_fit_df = pd.DataFrame([{
            "driver": EXPLODE_DRIVER, "mode": EXPLODE_MODE,
            "safe_levels": ",".join(str(v) for v in safe_levels),
            "target_explode_rate": TARGET_EXPLODE_RATE,
        }])
        explode_fit_df.to_csv(os.path.join(RESULT_DIR, "step11_explode_fit_params.csv"), index=False)
        level_stats.to_csv(os.path.join(RESULT_DIR, "step11b_explode_level_stats.csv"), index=False)

        plt.figure(figsize=(7, 5.5), dpi=140)
        bar_colors = ['#2ca02c' if s else '#d62728' for s in level_stats["safe"]]
        x_labels = [f"{v:g}" for v in level_stats[EXPLODE_DRIVER]]
        plt.bar(x_labels, level_stats["rate"], yerr=level_stats["sem"], color=bar_colors,
                alpha=0.8, capsize=6, edgecolor='black', linewidth=0.8)
        plt.axhline(TARGET_EXPLODE_RATE, color='#ff7f0e', linestyle='--', lw=1.5,
                    label=f'목표 Explode 비율 {TARGET_EXPLODE_RATE*100:.0f}%')
        for i, row in level_stats.iterrows():
            plt.text(i, row["rate"] + row["sem"] + 0.005, f"n={int(row['n'])}",
                      ha='center', fontsize=13)
        plt.title(f'{EXPLODE_DRIVER} 수준별 Explode(팽출) 비율 비교', fontsize=16, fontweight='bold')
        plt.xlabel(EXPLODE_DRIVER); plt.ylabel('Explode 비율')
        plt.ylim(0, float((level_stats["rate"] + level_stats["sem"]).max()) * 1.5 + 0.02)
        plt.grid(True, axis='y', linestyle=':', alpha=0.5)
        plt.legend(loc='upper left', fontsize=13, prop={'weight': 'bold'})
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURE_DIR, "explode_process_window.png"))
        plt.show()

    else:
        def explode_rate_model(X, X_mid, k):
            """X가 X_mid보다 커질수록 Explode 확률이 증가하는 증가형 시그모이드"""
            return 1.0 / (1.0 + np.exp(-k * (X - X_mid)))

        x_range = x.max() - x.min()
        x_mid0 = np.percentile(x, 85)
        k0 = 10.0 / x_range

        popt, pcov = curve_fit(
            explode_rate_model, x, y,
            p0=[x_mid0, k0],
            bounds=([x.min(), 1e-8], [x.max(), 50.0 / x_range]),
            maxfev=20000,
        )
        x_mid_opt, k_opt = popt
        x_mid_err = float(np.sqrt(pcov[0, 0]))
        reliability = x_mid_err / abs(x_mid_opt) if x_mid_opt != 0 else np.inf

        margin = np.log(1.0 / TARGET_EXPLODE_RATE - 1.0)
        x_safe_max = x_mid_opt - margin / k_opt
        J = np.array([1.0, margin / k_opt ** 2])
        x_safe_max_err = float(np.sqrt(J @ pcov @ J.T))

        verdict = "신뢰 가능" if reliability < 0.3 else "주의 (오차가 값의 30% 이상)"
        print(f"[Explode 상한 피팅] 기준 변수: {EXPLODE_DRIVER}")
        print(f"  X_mid = {x_mid_opt:.3e} ± {x_mid_err:.3e}  (오차비율 {reliability:.2f}, {verdict})")
        print(f"  안전 상한 (Explode<{TARGET_EXPLODE_RATE*100:.0f}%): {x_safe_max:.3e} ± {x_safe_max_err:.3e}")

        explode_fit_df = pd.DataFrame([{
            "driver": EXPLODE_DRIVER, "mode": EXPLODE_MODE,
            "x_mid_opt": x_mid_opt, "x_mid_err": x_mid_err,
            "slope_k_opt": k_opt, "safe_max": x_safe_max, "safe_max_err": x_safe_max_err,
            "target_explode_rate": TARGET_EXPLODE_RATE, "fit_reliability_ratio": reliability,
        }])
        explode_fit_df.to_csv(os.path.join(RESULT_DIR, "step11_explode_fit_params.csv"), index=False)

        xf = np.linspace(x.min(), x.max(), 300)
        bins_e = pd.qcut(x, q=8, duplicates="drop")
        binned_e = pd.DataFrame({EXPLODE_DRIVER: x, "is_explode": y, "bin": bins_e}) \
            .groupby("bin", observed=True) \
            .agg(x_mean=(EXPLODE_DRIVER, "mean"), n=("is_explode", "size"), rate=("is_explode", "mean")) \
            .reset_index()
        n_adj = binned_e["n"] + 4
        p_adj = (binned_e["rate"] * binned_e["n"] + 2) / n_adj
        binned_e["sem"] = np.sqrt(p_adj * (1 - p_adj) / n_adj)

        plt.figure(figsize=(9, 5.5), dpi=140)
        plt.errorbar(binned_e["x_mean"], binned_e["rate"], yerr=binned_e["sem"],
                     fmt='o', color='#1f77b4', ecolor='#d62728', capsize=4,
                     label='구간별 실측 (Mean ± SEM)')
        plt.plot(xf, explode_rate_model(xf, *popt), color='#ff7f0e', lw=2.5, label='증가형 로지스틱 피팅')
        plt.axvspan(x.min(), x_safe_max - x_safe_max_err, color='green', alpha=0.15,
                    label=f'안전 영역 (Explode<{TARGET_EXPLODE_RATE*100:.0f}%)')
        plt.axvline(x_safe_max, color='orange', linestyle='--', lw=1.5)
        plt.title(f'{EXPLODE_DRIVER} 에 따른 Explode(팽출) 확률 곡선', fontsize=16, fontweight='bold')
        plt.xlabel(EXPLODE_DRIVER); plt.ylabel('Explode 비율')
        plt.ylim(-0.05, 1.05)
        plt.grid(True, linestyle=':', alpha=0.6)
        plt.legend(loc='upper left', fontsize=13, prop={'weight': 'bold'})
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURE_DIR, "explode_process_window.png"))
        plt.show()


## B-6 — 완전한 공정 윈도우 가드레일: 하한 + 상한

B-3의 가드레일은 열입력 부족(Bad)만 방어했습니다. B-5에서 Explode의 유의미한 상한 요인을 찾았다면,
그 변수까지 함께 검사하는 확장 가드레일을 구성합니다. 요인을 못 찾았다면 기존 하한 전용 가드레일을 그대로
유지합니다(안전한 폴백). 상한 검사 대상 변수는 `EXPLODE_DRIVER`에 따라 자동으로 결정됩니다 — 어떤 변수가
채택되든 코드 수정 없이 대응합니다.

**출력 파일:** `result_rsw/step13_full_guardrail_log.csv`


In [ ]:
# ==========================================
# [Cell 14] 완전한 공정 윈도우 가드레일 (하한 + 상한)
# ==========================================
class FullProcessWindowGuardrail:
    """열입력 하한(미융착)과, 가능하면 Explode 상한까지 함께 검사하는 가드레일.
    Explode 기준 변수가 범주형이면 안전 수준 목록에 속하는지, 연속형이면 안전 상한을
    넘는지로 검사 방식이 갈린다 (Cell 13에서 결정된 EXPLODE_MODE를 그대로 따른다)."""
    def __init__(self, safe_heat_min: float, explode_driver: str = None,
                 explode_mode: str = None, explode_safe_max: float = None,
                 explode_safe_levels: list = None):
        self.safe_heat_min = safe_heat_min
        self.explode_driver = explode_driver
        self.explode_mode = explode_mode
        self.explode_safe_max = explode_safe_max
        self.explode_safe_levels = explode_safe_levels or []

    def validate(self, current: float, weld_time_s: float, **extra_params) -> dict:
        calc_heat = (current ** 2) * weld_time_s
        checks = []

        if calc_heat < self.safe_heat_min:
            corrected_time = self.safe_heat_min / (current ** 2)
            checks.append({
                "type": "UNDERHEAT", "status": "TRIPPED",
                "action": f"미융착 위험: 통전시간 보정 ({weld_time_s:.3f}s -> {corrected_time:.3f}s)",
            })
        else:
            checks.append({"type": "UNDERHEAT", "status": "PASS", "action": "정상 범위"})

        if self.explode_driver is not None:
            driver_value = extra_params.get(self.explode_driver)
            if driver_value is None:
                checks.append({
                    "type": "EXPLODE", "status": "SKIPPED",
                    "action": f"'{self.explode_driver}' 값이 입력되지 않아 상한 검사를 건너뜀",
                })
            elif self.explode_mode == "categorical":
                if driver_value not in self.explode_safe_levels:
                    checks.append({
                        "type": "EXPLODE", "status": "TRIPPED",
                        "action": f"과열/팽출 위험: {self.explode_driver}={driver_value:.3g}는 "
                                  f"안전 수준({self.explode_safe_levels})에 속하지 않음",
                    })
                else:
                    checks.append({"type": "EXPLODE", "status": "PASS", "action": "정상 범위"})
            elif driver_value > self.explode_safe_max:
                checks.append({
                    "type": "EXPLODE", "status": "TRIPPED",
                    "action": f"과열/팽출 위험: {self.explode_driver}={driver_value:.3g} > "
                              f"안전 상한 {self.explode_safe_max:.3g}",
                })
            else:
                checks.append({"type": "EXPLODE", "status": "PASS", "action": "정상 범위"})

        overall = "TRIPPED" if any(c["status"] == "TRIPPED" for c in checks) else "PASS"
        return {"input_heat": calc_heat, "checks": checks, "overall_status": overall}

defect_fit_for_guard = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]

if EXPLODE_DRIVER is not None:
    explode_fit_for_guard = pd.read_csv(os.path.join(RESULT_DIR, "step11_explode_fit_params.csv")).iloc[0]
    if explode_fit_for_guard['mode'] == 'categorical':
        safe_levels = [float(v) for v in str(explode_fit_for_guard['safe_levels']).split(',') if v != '']
        full_guardrail = FullProcessWindowGuardrail(
            safe_heat_min=defect_fit_for_guard['safe_heat_min'],
            explode_driver=explode_fit_for_guard['driver'],
            explode_mode='categorical', explode_safe_levels=safe_levels,
        )
        print(f"[가드레일 구성] 하한: 열입력 Q >= {defect_fit_for_guard['safe_heat_min']:.3e} "
              f"| 상한(범주형): {explode_fit_for_guard['driver']} 안전 수준={safe_levels}")
    else:
        full_guardrail = FullProcessWindowGuardrail(
            safe_heat_min=defect_fit_for_guard['safe_heat_min'],
            explode_driver=explode_fit_for_guard['driver'],
            explode_mode='continuous', explode_safe_max=explode_fit_for_guard['safe_max'],
        )
        print(f"[가드레일 구성] 하한: 열입력 Q >= {defect_fit_for_guard['safe_heat_min']:.3e} "
              f"| 상한: {explode_fit_for_guard['driver']} <= {explode_fit_for_guard['safe_max']:.3e}")
else:
    full_guardrail = FullProcessWindowGuardrail(safe_heat_min=defect_fit_for_guard['safe_heat_min'])
    print("[가드레일 구성] 하한만 적용 (Explode 유의 요인을 찾지 못해 상한 검사 생략)")

test_scenarios_full = [
    {"scenario": "정상 공정", "current": 2000.0, "weld_time_s": 0.8},
    {"scenario": "비정상 저열입력", "current": 900.0, "weld_time_s": 0.2},
]
if EXPLODE_DRIVER is not None:
    # 어떤 변수/모드가 채택되든 자동으로 "위험한 값" 시나리오를 구성한다
    if explode_fit_for_guard['mode'] == 'categorical':
        unsafe_candidates = [v for v in agg_explode[EXPLODE_DRIVER].unique() if v not in safe_levels]
        over_limit_value = float(unsafe_candidates[0]) if unsafe_candidates else float(agg_explode[EXPLODE_DRIVER].max())
    else:
        over_limit_value = explode_fit_for_guard['safe_max'] + explode_fit_for_guard['safe_max_err'] * 3 + 1.0
    test_scenarios_full.append({
        "scenario": f"비정상 {EXPLODE_DRIVER} 초과",
        "current": 2000.0, "weld_time_s": 0.8,
        EXPLODE_DRIVER: over_limit_value,
    })

log_full = []
for s in test_scenarios_full:
    extra = {k: v for k, v in s.items() if k not in ("scenario", "current", "weld_time_s")}
    r = full_guardrail.validate(s["current"], s["weld_time_s"], **extra)
    log_full.append({
        "scenario": s["scenario"], "overall_status": r["overall_status"],
        "input_heat": r["input_heat"],
        "detail": " | ".join(f"{c['type']}:{c['status']}" for c in r["checks"]),
    })

full_guard_df = pd.DataFrame(log_full)
full_guard_df.to_csv(os.path.join(RESULT_DIR, "step13_full_guardrail_log.csv"), index=False)
display(full_guard_df)


## B-7 — 딥러닝 대리모델 Stratified K-Fold 재검증

B-2의 `Val F1=0.857`은 단일 80/20 비층화 분할에서 나온 값입니다. 전체 Bad 샘플이 21개뿐이라 20% 검증셋에는
Bad가 약 4개만 들어가고, F1을 사실상 4개짜리 표본으로 계산한 셈이라 통계적으로 신뢰하기 어렵습니다.

여기서는 **Stratified 5-Fold 교차검증**으로 매 fold마다 새 모델을 처음부터 학습하고, fold별 F1의 평균±표준편차를
보고합니다. 또한 5개 fold를 합치면 **493개 샘플 전체(Bad 21개 포함)가 정확히 한 번씩 검증**되므로, 이 예측들을
모아 **풀링된 혼동행렬**도 함께 계산합니다 — 개별 fold보다 훨씬 안정적인 전체 성능 추정치입니다.

**출력 파일:** `result_rsw/step12_cv_fold_metrics.csv`, `result_rsw/step12b_cv_summary.csv`


In [ ]:
# ==========================================
# [Cell 15] 딥러닝 대리모델 Stratified K-Fold 재검증
# ==========================================
from sklearn.model_selection import StratifiedKFold

df_cv = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))
X_cv_raw = df_cv[feature_cols].values
y_cv_raw = df_cv['is_bad'].values

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

fold_metrics = []
pooled_preds, pooled_targets = [], []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_cv_raw, y_cv_raw), start=1):
    X_tr, X_va = X_cv_raw[train_idx], X_cv_raw[val_idx]
    y_tr, y_va = y_cv_raw[train_idx], y_cv_raw[val_idx]

    # 스케일러는 각 fold의 train만으로 적합 (검증 폴드 정보 유출 방지)
    x_min, x_max = X_tr.min(axis=0), X_tr.max(axis=0)
    X_tr_n = (X_tr - x_min) / (x_max - x_min + 1e-8)
    X_va_n = (X_va - x_min) / (x_max - x_min + 1e-8)

    tr_ds = TensorDataset(torch.tensor(X_tr_n, dtype=torch.float32),
                           torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1))
    va_ds = TensorDataset(torch.tensor(X_va_n, dtype=torch.float32),
                           torch.tensor(y_va, dtype=torch.float32).unsqueeze(1))
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False)

    fold_model = RSWDefectNet(input_dim=len(feature_cols)).to(DEVICE)
    fold_opt = optim.AdamW(fold_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    fold_sched = optim.lr_scheduler.CosineAnnealingLR(fold_opt, T_max=NUM_EPOCHS, eta_min=1e-6)
    criterion_cv = nn.BCELoss()

    for epoch in range(NUM_EPOCHS):
        fold_model.train()
        for bx, by in tr_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            fold_opt.zero_grad()
            loss = criterion_cv(fold_model(bx), by)
            loss.backward()
            fold_opt.step()
        fold_sched.step()

    fold_model.eval()
    va_preds, va_targets = [], []
    with torch.no_grad():
        for bx, by in va_loader:
            bx = bx.to(DEVICE)
            preds = fold_model(bx).cpu().numpy().flatten()
            va_preds.extend((preds >= 0.5).astype(float))
            va_targets.extend(by.numpy().flatten())

    fold_f1 = f1_score(va_targets, va_preds, zero_division=0)
    fold_acc = float(np.mean(np.array(va_preds) == np.array(va_targets))) * 100.0
    n_bad_fold = int(np.sum(np.array(va_targets) == 1))
    fold_metrics.append({"fold": fold_idx, "n_val": len(va_targets), "n_bad_val": n_bad_fold,
                          "f1": fold_f1, "accuracy": fold_acc})
    pooled_preds.extend(va_preds)
    pooled_targets.extend(va_targets)
    print(f"[Fold {fold_idx}] n_val={len(va_targets)} (Bad={n_bad_fold}) F1={fold_f1:.3f} Acc={fold_acc:.1f}%")

fold_df = pd.DataFrame(fold_metrics)
fold_df.to_csv(os.path.join(RESULT_DIR, "step12_cv_fold_metrics.csv"), index=False)

pooled_preds_arr, pooled_targets_arr = np.array(pooled_preds), np.array(pooled_targets)
pooled_f1 = f1_score(pooled_targets_arr, pooled_preds_arr, zero_division=0)
tp = int(np.sum((pooled_preds_arr == 1) & (pooled_targets_arr == 1)))
fp = int(np.sum((pooled_preds_arr == 1) & (pooled_targets_arr == 0)))
fn = int(np.sum((pooled_preds_arr == 0) & (pooled_targets_arr == 1)))
tn = int(np.sum((pooled_preds_arr == 0) & (pooled_targets_arr == 0)))

print(f"\n[요약] Fold별 F1 평균±표준편차: {fold_df['f1'].mean():.3f} ± {fold_df['f1'].std():.3f}")
print(f"[요약] Fold별 Accuracy 평균±표준편차: {fold_df['accuracy'].mean():.1f}% ± {fold_df['accuracy'].std():.1f}%")
print(f"[풀링된 혼동행렬] TP={tp} FP={fp} FN={fn} TN={tn}  (전체 493건, Bad {tp+fn}건이 정확히 한 번씩 검증됨)")
print(f"[풀링된 F1] {pooled_f1:.3f}  (기존 단일 분할 F1=0.857과 비교)")

cv_summary_df = pd.DataFrame([{
    "n_folds": N_FOLDS, "f1_mean": fold_df['f1'].mean(), "f1_std": fold_df['f1'].std(),
    "acc_mean": fold_df['accuracy'].mean(), "acc_std": fold_df['accuracy'].std(),
    "pooled_f1": pooled_f1, "pooled_tp": tp, "pooled_fp": fp, "pooled_fn": fn, "pooled_tn": tn,
}])
cv_summary_df.to_csv(os.path.join(RESULT_DIR, "step12b_cv_summary.csv"), index=False)


## B-8 — Phase 1: 실시간 온라인 학습 시뮬레이션

지난 제안(용접 요소·상관관계 → 실시간 파인튜닝 → 실시간 시각화)의 Phase 1을 이 노트북 안에서 데이터로
시뮬레이션합니다. 실제 센서/로봇이 없으므로, **493개 샘플을 `sample_id` 순서대로 하나씩 "도착"하는
스트림처럼 재생**하면서 지금까지 배치로 한 번에 했던 피팅을 주기적으로 재적합해 추정치가 어떻게
좁혀지는지 보여줍니다.

**중요한 전제 확인:** `sample_id` 순서가 실제 무작위가 아니라 **실험 진행 순서**라는 것을 먼저 데이터로
확인했습니다 — `Bad`는 초반 50건 안에 21건 중 17건이 몰려있고(저열입력 조건을 초반에 집중 시험한 것으로
추정), `Explode`는 300~400번째 구간에서 8건→24건으로 급증합니다(전극각도 15° 조건이 후반에 몰려있던
것으로 추정). 즉 이 재생 순서는 인위적으로 만든 게 아니라 **실제로 있었을 법한 온라인 학습 상황**을
그대로 반영합니다 — 초반에는 열입력 하한을 빠르게 학습하지만, 각도발 Explode 위험은 한동안 "보이지
않다가" 뒤늦게 드러나는 현실적인 문제가 재현됩니다.

**방법론적 한계 (명시):** 이것은 진짜 순차 베이지안 업데이트가 아니라, **일정 배치(25건)마다 지금까지
누적된 데이터 전체로 `curve_fit`을 다시 돌리는 "주기적 재적합" 방식의 근사**입니다. 실제 온라인 시스템은
칼만 필터나 순차 베이지안 갱신처럼 이전 추정치에 새 데이터만 반영하는 게 정석이지만, 이 노트북에서는
기존에 이미 검증된 배치 피팅 코드를 그대로 재사용하기 위해 이 근사를 택했습니다.

**출력 파일:** `result_rsw/step15_online_convergence_log.csv`, `result_rsw/figures/online_convergence_summary.png`


In [ ]:
# ==========================================
# [Cell 17] Phase 1 — 스트리밍 재생 설정 및 재적합 함수
# ==========================================
import time
from IPython.display import clear_output

stream_df = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv")) \
    .sort_values("sample_id").reset_index(drop=True)

BATCH_SIZE_STREAM = 25
REPLAY_DELAY_SEC = 0.15   # 프레임 간 대기 (데모 시연용 — 0으로 두면 즉시 진행)
MIN_BAD_FOR_FIT = 3        # Bad/Good 각각 최소 이 개수 이상이어야 재적합 시도

def try_fit_bad_threshold(df_so_far: pd.DataFrame):
    """지금까지 누적된 데이터로 Bad 하한 로지스틱을 재적합. 데이터 부족하면 None."""
    n_bad = df_so_far["is_bad"].sum()
    n_good = len(df_so_far) - n_bad
    if n_bad < MIN_BAD_FOR_FIT or n_good < MIN_BAD_FOR_FIT:
        return None
    x = df_so_far["heat_input_proxy"].values.astype(float)
    y = df_so_far["is_bad"].values.astype(float)
    x_range = x.max() - x.min()
    if x_range <= 0:
        return None
    try:
        popt, pcov = curve_fit(
            ResistanceWeldingOptimizer.bad_rate_logistic_model, x, y,
            p0=[np.percentile(x, 15), 10.0 / x_range],
            bounds=([x.min(), 1e-8], [x.max(), 50.0 / x_range]),
            maxfev=20000,
        )
        return float(popt[0]), float(np.sqrt(pcov[0, 0]))
    except Exception:
        return None

def angle_level_rates(df_so_far: pd.DataFrame) -> pd.DataFrame:
    """angle_deg 수준별 누적 Explode 비율 ± Agresti-Coull SEM (관측된 수준만 포함)."""
    rows = []
    for lvl in sorted(df_so_far["angle_deg"].unique()):
        sub = df_so_far[df_so_far["angle_deg"] == lvl]
        n = len(sub)
        k = sub["is_explode"].sum()
        n_adj = n + 4
        p_adj = (k + 2) / n_adj
        sem = np.sqrt(p_adj * (1 - p_adj) / n_adj)
        rows.append({"angle_deg": lvl, "n": n, "rate": k / n, "sem": sem})
    return pd.DataFrame(rows)

convergence_log = []
print(f"[준비 완료] 총 {len(stream_df)}건을 {BATCH_SIZE_STREAM}건씩 배치로 재생합니다.")


In [ ]:
# ==========================================
# [Cell 17-2] Phase 1 — 실시간 시각화 루프
# ==========================================
n_total = len(stream_df)
batch_ends = list(range(BATCH_SIZE_STREAM, n_total + 1, BATCH_SIZE_STREAM))
if batch_ends[-1] != n_total:
    batch_ends.append(n_total)

for cutoff in batch_ends:
    df_so_far = stream_df.iloc[:cutoff]
    batch_new = stream_df.iloc[max(0, cutoff - BATCH_SIZE_STREAM):cutoff]

    fit_result = try_fit_bad_threshold(df_so_far)
    angle_rates = angle_level_rates(df_so_far)

    log_row = {"n_seen": cutoff,
               "q_min_opt": fit_result[0] if fit_result else np.nan,
               "q_min_err": fit_result[1] if fit_result else np.nan}
    for _, r in angle_rates.iterrows():
        log_row[f"angle{int(r['angle_deg'])}_rate"] = r["rate"]
    convergence_log.append(log_row)

    clear_output(wait=True)
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=120)

    ax = axes[0]
    ax.scatter(df_so_far["heat_input_proxy"], df_so_far["is_bad"], s=10, alpha=0.3,
               color='#1f77b4', label=f'누적 관측 (n={cutoff})')
    ax.scatter(batch_new["heat_input_proxy"], batch_new["is_bad"], s=30, alpha=0.9,
               color='#d62728', label=f'이번 배치 신규 (n={len(batch_new)})')
    if fit_result:
        q_min_opt, q_min_err = fit_result
        ax.axvline(q_min_opt, color='green', lw=2, label=f'Q_min≈{q_min_opt:.2e}±{q_min_err:.2e}')
        ax.axvspan(q_min_opt - q_min_err, q_min_opt + q_min_err, color='green', alpha=0.15)
    else:
        ax.text(0.5, 0.5, "데이터 부족\n(재적합 대기 중)", transform=ax.transAxes,
                ha='center', fontsize=15, color='gray')
    ax.set_xlabel('열입력 proxy Q'); ax.set_ylabel('is_bad')
    ax.set_title(f'실시간 Bad 하한 추정 ({cutoff}/{n_total}건 관측)', fontweight='bold')
    ax.legend(fontsize=11, loc='center right', prop={'weight': 'bold'})

    ax2 = axes[1]
    colors = ['#2ca02c' if r < TARGET_EXPLODE_RATE else '#d62728' for r in angle_rates["rate"]]
    ax2.bar(angle_rates["angle_deg"].astype(int).astype(str), angle_rates["rate"],
            yerr=angle_rates["sem"], color=colors, alpha=0.8, capsize=6, edgecolor='black')
    ax2.axhline(TARGET_EXPLODE_RATE, color='orange', ls='--', label='목표 5%')
    for i, row in angle_rates.reset_index(drop=True).iterrows():
        ax2.text(i, row["rate"] + row["sem"] + 0.01, f"n={int(row['n'])}", ha='center', fontsize=13)
    ax2.set_ylim(0, 0.3)
    ax2.set_title('실시간 각도별 Explode 비율', fontweight='bold')
    ax2.set_xlabel('angle_deg'); ax2.set_ylabel('Explode 비율')
    ax2.legend(fontsize=11, prop={'weight': 'bold'})

    plt.tight_layout()
    plt.show()
    time.sleep(REPLAY_DELAY_SEC)

convergence_df = pd.DataFrame(convergence_log)
convergence_df.to_csv(os.path.join(RESULT_DIR, "step15_online_convergence_log.csv"), index=False)
print(f"[완료] {n_total}건 스트리밍 재생 종료. 수렴 로그 저장: step15_online_convergence_log.csv")


# C. 데이터셋 시각화

물리 모델링에 들어가기 전에, 원자료 자체의 분포를 먼저 확인합니다.

## 데이터셋 출처
- **출처:** [Resistance Spot Welding Insights — Mendeley Data](https://data.mendeley.com/datasets/rwh8kjzdch/3) (CC BY 4.0)
- **파일:** `Data_RSW.csv` (495개 용접 샘플, 샘플당 여러 행의 통전 중 힘(Force) 시계열 포함) + IR 열화상 이미지 99장
- **다운로드:** 루트의 `download_rsw.py` 실행 (Kaggle 계정/토큰 불필요, Mendeley 공개 API 직접 호출)


## C-1 — 데이터셋 현황 및 IR 열화상 이미지 미리보기

정형 데이터(전류·시간·압력 등)와 함께 제공되는 **IR 열화상 이미지**를 함께 확인하여, 이 데이터셋이 실제로 멀티모달(수치+이미지) 성격을 갖는지 검증합니다.


In [ ]:
# ==========================================
# [Cell EDA] 데이터셋 현황 시각화 + IR 이미지 샘플
# ==========================================
df_eda = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))

fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=120)
fig.suptitle(f'RSW 데이터셋 현황  |  유효 샘플 {len(df_eda):,}개', fontsize=18, fontweight='bold')

cat_colors = {'Good': '#2ca02c', 'Bad': '#d62728', 'Explode': '#ff7f0e'}
num_features = ['avg_current_A', 'weld_time_s', 'pressure_psi',
                 'heat_input_proxy', 'nugget_diameter_mm', 'pull_test_N']
for i, col in enumerate(num_features):
    ax = axes[i // 3, i % 3]
    for cat, color in cat_colors.items():
        data = df_eda[df_eda['category'] == cat][col].dropna()
        ax.hist(data, bins=20, alpha=0.55, color=color, label=cat, density=True)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.tick_params(labelsize=7)
    if i == 0:
        ax.legend(fontsize=10, prop={'weight': 'bold'})

fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "eda_overview.png"), bbox_inches='tight')
plt.show()

# 카테고리 분포
counts = df_eda['category'].value_counts()
category_dist_df = counts.rename_axis('category').reset_index(name='count')
category_dist_df['percentage'] = category_dist_df['count'] / len(df_eda) * 100
category_dist_df.to_csv(os.path.join(RESULT_DIR, "step0_category_distribution.csv"), index=False)
print("[카테고리 분포]")
print(counts.to_string())
print(f"Bad 비율: {counts.get('Bad', 0) / len(df_eda) * 100:.1f}%  |  "
      f"Explode 비율: {counts.get('Explode', 0) / len(df_eda) * 100:.1f}%")

# IR 이미지 샘플 몇 장 미리보기 (파일명 IR_<번호>.jpg 형태, 전체 495건 중 일부만 존재)
ir_files = sorted(os.listdir(IMG_DIR))[:6] if os.path.isdir(IMG_DIR) else []
if ir_files:
    fig2, axes2 = plt.subplots(1, len(ir_files), figsize=(3 * len(ir_files), 3), dpi=100)
    fig2.suptitle('IR 열화상 이미지 샘플 (멀티모달 데이터 확인)', fontsize=15, fontweight='bold')
    for ax, fname in zip(axes2, ir_files):
        img = plt.imread(os.path.join(IMG_DIR, fname))
        ax.imshow(img)
        ax.set_title(fname, fontsize=11)
        ax.axis('off')
    fig2.tight_layout()
    fig2.savefig(os.path.join(FIGURE_DIR, "ir_sample_preview.png"), bbox_inches='tight')
    plt.show()
else:
    print("[알림] ir_images 폴더가 비어 있습니다. download_rsw.py를 먼저 실행하세요.")


## C-2 — 데이터셋 전체 요소 분포 (종합 EDA)

`C-1`는 주요 6개 변수만 다뤘습니다. 여기서는 `step1_aggregated_samples.csv`의 **의미 있는 모든 열**을
빠짐없이 시각화합니다. **5행 × 2열**로 배치하고, 연관 있는 변수끼리 같은 행에 오도록 순서를 정했습니다.

| 행 | 배치 | 연관 근거 |
|---|---|---|
| 1 | `avg_current_A`, `peak_current_A` | 같은 전류 계측치(평균/피크) |
| 2 | `weld_time_s`, `heat_input_proxy` | $Q \propto I^2 t$ — 통전시간이 열입력의 구성요소 |
| 3 | `nugget_diameter_mm`, `pull_test_N` | D-5에서 검증된 상관관계($r=0.535$) |
| 4 | `pressure_psi`, `angle_deg` | 둘 다 실험설계(DOE) 공정 설정 변수 |
| 5 | `thickness_avg_mm`, `category` | 나머지 |

모든 패널에 x축·y축 라벨과 범례(우측 상단)를 표시합니다.

- **연속형 (히스토그램):** `avg_current_A`, `peak_current_A`, `weld_time_s`, `heat_input_proxy`,
  `nugget_diameter_mm`, `pull_test_N`, `thickness_avg_mm`
- **이산형/범주형 (수준별 막대):** `pressure_psi`(4수준), `angle_deg`(2수준) — 값이 몇 개뿐인 실험설계
  변수라 히스토그램보다 수준별 개수 막대가 더 적절합니다 (B-5에서 `angle_deg`에 `qcut`을 잘못
  적용했다가 겪은 문제와 같은 이유)
- **category 자체 분포:** Good/Bad/Explode 개수

**의도적으로 제외한 열:** `sample_id`(단순 인덱스), `is_bad`·`is_explode`(`category`에서 파생된 이진
지표라 `category` 분포와 정보가 중복됨).

**출력 파일:** `result_rsw/figures/full_dataset_distribution.png`


In [ ]:
# ==========================================
# [Cell 21] 데이터셋 전체 요소 분포 (종합 EDA)
# ==========================================
df_full = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))

# (열 이름, 유형, x축 라벨) — 연관 있는 변수끼리 같은 행(2개씩)에 오도록 순서 배치
panel_specs = [
    ('avg_current_A', 'hist', '평균 전류 (A)'),
    ('peak_current_A', 'hist', '피크 전류 (A)'),
    ('weld_time_s', 'hist', '통전 시간 (s)'),
    ('heat_input_proxy', 'hist', r'열입력 proxy $Q \propto I^2 t$'),
    ('nugget_diameter_mm', 'hist', '너겟 지름 (mm)'),
    ('pull_test_N', 'hist', '인장강도 (N)'),
    ('pressure_psi', 'bar', '가압력 (PSI)'),
    ('angle_deg', 'bar', '전극각도 (°)'),
    ('thickness_avg_mm', 'hist', '판재 두께 (mm)'),
    ('category', 'catbar', '결함 카테고리'),
]

n_cols = 2
n_rows = int(np.ceil(len(panel_specs) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(11, 3.6 * n_rows), dpi=110)
axes = axes.flatten()

for ax, (col, kind, xlabel) in zip(axes, panel_specs):
    if kind == 'hist':
        for cat, color in cat_colors.items():
            data = df_full[df_full['category'] == cat][col].dropna()
            ax.hist(data, bins=20, alpha=0.55, color=color, label=cat, density=True)
        ax.set_ylabel('밀도 (Density)', fontsize=11)
    elif kind == 'bar':
        levels = sorted(df_full[col].unique())
        width = 0.25
        for i, (cat, color) in enumerate(cat_colors.items()):
            counts = [len(df_full[(df_full[col] == lvl) & (df_full['category'] == cat)]) for lvl in levels]
            x_pos = np.arange(len(levels)) + (i - 1) * width
            ax.bar(x_pos, counts, width=width, color=color, alpha=0.85, label=cat)
        ax.set_xticks(np.arange(len(levels)))
        ax.set_xticklabels([f"{v:g}" for v in levels], fontsize=11)
        ax.set_ylabel('개수 (Count)', fontsize=11)
    else:  # catbar
        cat_counts = df_full['category'].value_counts()
        for cat in cat_counts.index:
            v = cat_counts[cat]
            ax.bar(cat, v, color=cat_colors[cat], alpha=0.85, label=cat)
            ax.text(cat, v + 5, str(v), ha='center', fontsize=13)
        ax.set_ylabel('개수 (Count)', fontsize=11)

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_title(col, fontsize=14, fontweight='bold')
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=10, loc='upper right', prop={'weight': 'bold'})

fig.suptitle(f'RSW 데이터셋 전체 요소 분포 (n={len(df_full)})', fontsize=19, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "full_dataset_distribution.png"))
plt.show()

print(f"[완료] {len(panel_specs)}개 패널, {n_rows}행×{n_cols}열 배치 (연관 변수는 같은 행에 배치)")


# D. 데이터 시각화 및 해석

섹션 B에서 구현한 각 모델의 피팅 결과를 그래프로 확인하고 해석합니다.


## D-1 — 미융착(Bad) 확률 곡선 피팅 및 최소 안전 열입력

**쉽게 말하면:** 열입력이 낮을수록 접합이 덜 되어 미융착이 생길 확률이 높아지고, 열입력이 충분히 크면 그 확률이 0에 가까워지는 S자 모양 곡선을 실측 데이터에 맞춥니다. 이 곡선이 50%를 지나는 지점이 아래 $Q_{min}$ 입니다.

**피팅 함수 — 감소형 로지스틱:**

$$p(Q) = \frac{1}{1 + e^{\,k(Q - Q_{min})}}$$

- $p(Q)$: 열입력 $Q$에서 미융착(Bad)이 발생할 확률
- $Q_{min}$: $p(Q_{min})=0.5$가 되는 문턱 열입력 (로지스틱 중심점)
- $k$: 전이 기울기. $k$가 클수록 $Q_{min}$ 부근에서 확률이 더 가파르게 떨어짐
- $Q \to \infty$ 이면 $p \to 0$(열입력이 충분하면 안전), $Q \to -\infty$ 이면 $p \to 1$(열입력 부족 시 결함) —
  Bad는 "열입력이 낮을 때" 증가하는 결함이므로 지수 항의 부호가 일반 로지스틱과 반대(감소형)입니다.
- 이 함수를 원본 493개 샘플의 0/1 라벨에 `scipy.optimize.curve_fit`으로 직접 피팅해 $Q_{min}$, $k$를 추정하고,
  목표 Bad 비율(5%)이 되는 지점을 역산해 안전 열입력 하한을 구합니다.

**그래프 구성:**

| 요소 | 의미 |
|------|------|
| 파란 점 + 에러바 | 구간별 실측 Bad 비율과 Agresti-Coull SEM |
| 빨간 실선 | 피팅된 감소형 로지스틱 곡선 |
| 초록 음영 영역 | Bad 비율 5% 미만을 보장하는 **안전 공정 윈도우** ($Q \geq Q_{min,safe}$) |

**출력 파일:** `result_rsw/figures/defect_process_window.png`


In [ ]:
# ==========================================
# [Cell 4] 미융착 확률 곡선 시각화
# ==========================================
binned = pd.read_csv(os.path.join(RESULT_DIR, "step2_binned_stats.csv"))
fit = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]
agg = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))

q_min, q_min_err = fit['q_min_opt'], fit['q_min_err']
k_opt, safe_q_min = fit['slope_k_opt'], fit['safe_heat_min']

x_axis = np.linspace(agg['heat_input_proxy'].min(), agg['heat_input_proxy'].max(), 300)
y_fitted = 1.0 / (1.0 + np.exp(k_opt * (x_axis - q_min)))

plt.figure(figsize=(10, 6), dpi=150)
plt.errorbar(binned['heat_mean'], binned['bad_rate'], yerr=binned['bad_rate_sem'],
             fmt='o', color='#1f77b4', ecolor='#d62728', elinewidth=2, capsize=5,
             label='구간별 Bad 비율 (Mean ± SEM, Agresti-Coull)')
plt.plot(x_axis, y_fitted, color='#2ca02c', lw=2.5,
         label=rf'로지스틱 피팅 ($Q_{{min}}={q_min:.2e} \pm {q_min_err:.2e}$)')
plt.axvspan(safe_q_min, agg['heat_input_proxy'].max(), color='green', alpha=0.15,
            label=f'안전 공정 윈도우 (Bad 비율 < {fit["target_bad_rate"]*100:.0f}%)')
plt.axvline(safe_q_min, color='green', linestyle='--', lw=1.5)

plt.title('열입력(줄열 proxy)에 따른 미융착(Bad) 확률 곡선', fontsize=16, fontweight='bold')
plt.xlabel(r'열입력 proxy $Q \propto I_{avg}^2 \cdot t$', fontsize=15)
plt.ylabel('Bad 비율', fontsize=15)
plt.ylim(-0.05, 1.05)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='upper right', frameon=True, prop={'weight': 'bold'})

# 피팅 신뢰도 검증 — 이전 노트북의 "항상 참인 해석" 버그 방지
reliability = fit['fit_reliability_ratio']
verdict = "신뢰 가능" if reliability < 0.3 else "주의 (오차가 값의 30% 이상)"
plt.gca().text(
    0.5, 0.5,
    f"[피팅 신뢰도] q_min_err/q_min = {reliability:.2f} → {verdict}",
    transform=plt.gca().transAxes, fontsize=13, fontweight="bold",
    ha='center', va='center',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, pad=0.6)
)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "defect_process_window.png"))
plt.show()
print(f"[신뢰도] q_min_err/q_min = {reliability:.3f} ({verdict})")


## D-2 — 너겟 지름 성장 곡선 피팅

단일 포인트가 아닌, 열입력 구간별 **평균 너겟 지름과 SEM**을 포화 성장 모델(saturating exponential)로 피팅합니다.

$$D(Q) = D_0 + (D_{max} - D_0)(1 - e^{-Q/\tau})$$

**출력 파일:** `result_rsw/figures/nugget_growth_curve.png`


In [ ]:
# ==========================================
# [Cell 5] 너겟 지름 성장 곡선 시각화
# ==========================================
nfit = pd.read_csv(os.path.join(RESULT_DIR, "step4_nugget_fit_params.csv")).iloc[0]
d0, dmax, tau = nfit['D0_opt'], nfit['Dmax_opt'], nfit['tau_opt']

x_axis = np.linspace(agg['heat_input_proxy'].min(), agg['heat_input_proxy'].max(), 300)
y_fitted = d0 + (dmax - d0) * (1.0 - np.exp(-x_axis / tau))

plt.figure(figsize=(10, 6), dpi=150)
plt.errorbar(binned['heat_mean'], binned['nugget_mean'], yerr=binned['nugget_sem'],
             fmt='o', color='#1f77b4', ecolor='#d62728', elinewidth=2, capsize=5,
             label='구간별 평균 너겟 지름 (Mean ± SEM)')
plt.plot(x_axis, y_fitted, color='#9467bd', lw=2.5,
         label=rf'포화 성장 피팅 ($D_{{max}}={dmax:.2f} \pm {nfit["Dmax_err"]:.2f}$ mm)')

plt.title('열입력에 따른 너겟 지름 성장 곡선', fontsize=16, fontweight='bold')
plt.xlabel(r'열입력 proxy $Q \propto I_{avg}^2 \cdot t$', fontsize=15)
plt.ylabel('너겟 지름 (mm)', fontsize=15)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', frameon=True, prop={'weight': 'bold'})
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "nugget_growth_curve.png"))
plt.show()


## D-3 — 학습 이력 시각화


In [ ]:
# ==========================================
# [Cell 7] 학습 이력 시각화
# ==========================================
from matplotlib.ticker import MaxNLocator

hist_df = pd.read_csv(os.path.join(RESULT_DIR, "step5_training_history.csv"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=130)

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train Loss', color='#1f77b4',
             marker='o', markersize=4)
axes[0].plot(hist_df['epoch'], hist_df['val_loss'], label='Val Loss', color='#d62728', ls='--',
             marker='o', markersize=4)
axes[0].set_title('BCE Loss 추이', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(prop={'weight': 'bold'}); axes[0].grid(True, ls=':', alpha=0.6)
axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))

# 왼쪽 축=F1(초록), 오른쪽 축=Accuracy(보라)로 데이터-축 매핑은 맞으나,
# 기존 코드는 이 서브플롯에 legend()를 호출하지 않아 범례가 아예 안 그려지고 있었다.
# twinx()로 분리된 두 Axes는 legend()를 각각 호출해도 서로의 라인을 모르므로 handle을 직접 합친다.
line_f1 = axes[1].plot(hist_df['epoch'], hist_df['val_f1_score'], label='Val F1', color='#2ca02c',
                        marker='o', markersize=4)
ax2 = axes[1].twinx()
line_acc = ax2.plot(hist_df['epoch'], hist_df['val_accuracy'], label='Val Acc', color='#9467bd', ls='--',
                     marker='o', markersize=4)
axes[1].set_title('Val F1 & Accuracy', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1', color='#2ca02c')
ax2.set_ylabel('Accuracy (%)', color='#9467bd')
axes[1].grid(True, ls=':', alpha=0.6)
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
lines = line_f1 + line_acc
axes[1].legend(lines, [l.get_label() for l in lines], loc='center right', prop={'weight': 'bold'})

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "training_history.png"))
plt.show()


## D-4 — 열화상 특징과 열입력·너겟지름 상관분석

`step1_load_and_aggregate()` 결과(`agg`)와 IR 특징(`thermal_df`)을 `sample_id`로 병합한 뒤,
Pearson 상관계수와 p-value를 함께 보고합니다. `fit_reliability_ratio`와 같은 맥락으로,
**숫자만 보고 "상관 있다/없다"를 판단하지 않고 유의성 기준(p<0.05)을 명시**합니다.

**출력 파일:** `result_rsw/step8_thermal_correlations.csv`, `result_rsw/figures/thermal_feature_correlation.png`


In [ ]:
# ==========================================
# [Cell 11] 열화상 특징 상관분석 시각화
# ==========================================
from scipy.stats import pearsonr

merged = thermal_df.merge(agg, on="sample_id", how="inner")

repr_check_df = pd.DataFrame([{
    "n_total_samples": len(agg), "n_image_samples": len(merged),
    "total_heat_min": agg['heat_input_proxy'].min(), "total_heat_max": agg['heat_input_proxy'].max(),
    "image_subset_heat_min": merged['heat_input_proxy'].min(),
    "image_subset_heat_max": merged['heat_input_proxy'].max(),
}])
repr_check_df.to_csv(os.path.join(RESULT_DIR, "step8b_representativeness_check.csv"), index=False)
merged['category'].value_counts().rename_axis('category').reset_index(name='count').to_csv(
    os.path.join(RESULT_DIR, "step8c_image_subset_category_distribution.csv"), index=False)

print(f"[대표성 점검] 전체 유효 샘플 {len(agg)}건 중 IR 이미지 매칭 {len(merged)}건")
print(f"  - 전체 열입력 범위      : {agg['heat_input_proxy'].min():.2e} ~ {agg['heat_input_proxy'].max():.2e}")
print(f"  - 이미지 서브셋 열입력 범위: {merged['heat_input_proxy'].min():.2e} ~ {merged['heat_input_proxy'].max():.2e}")
print(f"  - 이미지 서브셋 카테고리 분포:")
print(merged['category'].value_counts().to_string())

pairs = [
    ("hot_area_fraction", "heat_input_proxy", "고온영역 비율", r"열입력 proxy $Q$"),
    ("hot_area_fraction", "nugget_diameter_mm", "고온영역 비율", "너겟 지름 (mm)"),
    ("hotspot_aspect_ratio", "nugget_diameter_mm", "고온영역 종횡비", "너겟 지름 (mm)"),
]

corr_rows = []
for xcol, ycol, _, _ in pairs:
    sub = merged[[xcol, ycol]].dropna()
    r, p = pearsonr(sub[xcol], sub[ycol])
    corr_rows.append({"x": xcol, "y": ycol, "n": len(sub), "pearson_r": r, "p_value": p})

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(os.path.join(RESULT_DIR, "step8_thermal_correlations.csv"), index=False)
display(corr_df)

fig, axes = plt.subplots(1, len(pairs), figsize=(5 * len(pairs), 4.5), dpi=130)
for ax, (xcol, ycol, xlabel, ylabel), row in zip(axes, pairs, corr_rows):
    sub = merged[[xcol, ycol]].dropna()
    ax.scatter(sub[xcol], sub[ycol], s=20, alpha=0.6, color='#1f77b4')
    verdict = "유의미 (p<0.05)" if row["p_value"] < 0.05 else "유의성 부족 (p≥0.05)"
    ax.set_title(f"r={row['pearson_r']:.2f}, n={row['n']}\n{verdict}", fontsize=14)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(True, linestyle=':', alpha=0.5)

fig.suptitle("IR 열화상 물리 특징 ↔ 공정변수 상관관계 (n=99, 표본 편향 있음 — 참고용)",
             fontsize=15, fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(FIGURE_DIR, "thermal_feature_correlation.png"))
plt.show()

print()
print("[해석 시 주의] 이미지가 있는 샘플은 Sample ID 10~18, 100~189 구간에 몰려 있어 무작위 표본이 아닙니다.")
print("따라서 위 상관계수는 전체 493개 샘플의 모집단 상관관계를 대표하지 않을 수 있으며, 탐색적 결과로만 활용해야 합니다.")


## D-5 — 서두 상관계수 주장 검증

이 노트북 맨 위 마크다운에서 인용한 상관계수 3개(통전시간↔너겟지름 0.47, 줄열 proxy↔너겟지름 0.36,
너겟지름↔인장강도 0.53)는 지금까지 노트북 어떤 코드로도 재현되지 않은 **사전 검증 결과 텍스트**였습니다.
재현성을 위해 여기서 `step1_aggregated_samples.csv`로 직접 계산해 대조합니다.

**출력 파일:** `result_rsw/step9_intro_claims_check.csv`


In [ ]:
# ==========================================
# [Cell 12] 서두 상관계수 주장 검증
# ==========================================
from scipy.stats import pearsonr

claims = [
    ("weld_time_s", "nugget_diameter_mm", "통전시간 ↔ 너겟지름", 0.47),
    ("heat_input_proxy", "nugget_diameter_mm", "줄열 proxy(I²t) ↔ 너겟지름", 0.36),
    ("nugget_diameter_mm", "pull_test_N", "너겟지름 ↔ 인장강도(PullTest)", 0.53),
]

agg_check = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))

claim_rows = []
for xcol, ycol, label, claimed_r in claims:
    r, p = pearsonr(agg_check[xcol], agg_check[ycol])
    claim_rows.append({
        "관계": label, "주장된_r": claimed_r, "실측_r": round(r, 3),
        "p_value": p, "일치_여부": "일치" if abs(r - claimed_r) < 0.05 else "불일치",
    })

claim_check_df = pd.DataFrame(claim_rows)
claim_check_df.to_csv(os.path.join(RESULT_DIR, "step9_intro_claims_check.csv"), index=False)
display(claim_check_df)


## D-6 — Phase 1: 수렴 추이 정적 요약

실시간 애니메이션을 직접 보지 않아도, 관측 샘플 수가 늘어남에 따라 추정치가 어떻게 좁혀졌는지 정적
그래프로 남깁니다. 최종(493건 배치) 피팅값과 비교해 온라인 추정이 결국 같은 값에 수렴하는지 확인합니다.


In [ ]:
# ==========================================
# [Cell 18] Phase 1 — 수렴 추이 요약 (정적)
# ==========================================
final_q_min = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]["q_min_opt"]

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=130)

ax = axes[0]
valid = convergence_df.dropna(subset=["q_min_opt"])
ax.errorbar(valid["n_seen"], valid["q_min_opt"], yerr=valid["q_min_err"],
            fmt='o-', capsize=3, color='#2ca02c', label='온라인 추정치')
ax.axhline(final_q_min, color='gray', ls=':', label='최종(493건) 배치 피팅값')
ax.set_xlabel('누적 관측 샘플 수'); ax.set_ylabel('Q_min 추정치')
ax.set_title('Bad 하한 추정치의 수렴 추이', fontweight='bold')
ax.legend(fontsize=11, prop={'weight': 'bold'})
ax.grid(True, ls=':', alpha=0.5)

ax2 = axes[1]
if "angle0_rate" in convergence_df.columns:
    ax2.plot(convergence_df["n_seen"], convergence_df["angle0_rate"], 'o-', color='#2ca02c', label='angle=0')
if "angle15_rate" in convergence_df.columns:
    ax2.plot(convergence_df["n_seen"], convergence_df["angle15_rate"], 'o-', color='#d62728', label='angle=15')
ax2.axhline(TARGET_EXPLODE_RATE, color='orange', ls='--', label='목표 5%')
ax2.set_xlabel('누적 관측 샘플 수'); ax2.set_ylabel('Explode 비율')
ax2.set_title('각도별 Explode 비율 수렴 추이', fontweight='bold')
ax2.legend(fontsize=11, prop={'weight': 'bold'})
ax2.grid(True, ls=':', alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "online_convergence_summary.png"))
plt.show()


## D-7 — 이미지 산점도: IR 열화상과 상관관계 그래프를 한 화면에서 대조

`D-4`의 상관분석은 `hot_area_fraction`(고온영역 비율)과 열입력 사이에 $r=0.41$의 유의미한 상관관계를
보였지만, 수치 산점도만으로는 "실제로 이미지가 어떻게 다르게 보이는지"를 확인할 수 없습니다. 여기서는
산점도의 점을 **실제 IR 썸네일 이미지로 대체**해, 그래프를 읽는 동시에 이미지 내용을 검증할 수 있게
합니다.

- x축(열입력 $Q$)·y축(고온영역 비율) 좌표는 그대로 유지하고, 그 자리에 해당 샘플의 실제 IR 이미지를 얹습니다.
- 전체 99장을 다 얹으면 겹쳐서 알아보기 어려우므로, 값 순으로 정렬한 뒤 등간격 인덱스로 최대 18장만
  서브샘플링합니다 (전체 분포 맥락은 배경의 옅은 점으로 유지).
- 썸네일 테두리 색으로 결함 카테고리(Good/Bad/Explode)를 함께 표시합니다.
- 이미지는 `B-4`에서 정의한 `crop_above_scalebar()`로 축척바를 제거한 뒤 사용합니다.

**한계:** `zoom` 값은 렌더링 결과를 보면서 조정이 필요할 수 있습니다 (이 노트북 작성 환경에서는 직접
렌더링해 확인할 수 없었습니다).

**출력 파일:** `result_rsw/figures/thermal_image_scatter.png`


In [ ]:
# ==========================================
# [Cell 19] 이미지 산점도 — IR 썸네일을 산점도 좌표에 직접 배치
# ==========================================
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

N_THUMBS = 18
IMG_ZOOM = 0.13
X_COL, Y_COL = "heat_input_proxy", "hot_area_fraction"

plot_df = merged.dropna(subset=[X_COL, Y_COL]).sort_values(X_COL).reset_index(drop=True)
# 값 순으로 정렬 후 등간격 인덱스로 서브샘플링 (전체 범위를 고르게 대표, 겹침 방지)
thumb_idx = np.linspace(0, len(plot_df) - 1, min(N_THUMBS, len(plot_df))).astype(int)
thumb_rows = plot_df.iloc[thumb_idx]

fig, ax = plt.subplots(figsize=(13, 9), dpi=120)

# 배경: 전체 99개 점을 옅게 표시해 전체 분포 맥락 유지
for cat, color in cat_colors.items():
    sub = plot_df[plot_df["category"] == cat]
    ax.scatter(sub[X_COL], sub[Y_COL], s=25, color=color, alpha=0.35, label=cat, zorder=1)

# 전경: 서브샘플링된 점만 실제 IR 썸네일로 대체 (테두리 색 = 결함 카테고리)
n_placed = 0
for _, row in thumb_rows.iterrows():
    sid = int(row["sample_id"])
    fname = os.path.join(IMG_DIR, f"IR_{sid}.jpg")
    if not os.path.exists(fname):
        continue
    img = plt.imread(fname)
    cropped_img, _ = crop_above_scalebar(img)
    thumb = OffsetImage(cropped_img, zoom=IMG_ZOOM)
    border_color = cat_colors.get(row["category"], "black")
    ab = AnnotationBbox(thumb, (row[X_COL], row[Y_COL]), frameon=True, pad=0.05,
                         bboxprops=dict(edgecolor=border_color, linewidth=2))
    ax.add_artist(ab)
    n_placed += 1

ax.set_xlabel(r'열입력 proxy $Q$'); ax.set_ylabel('고온영역 비율 (hot_area_fraction)')
ax.set_title(f'이미지 산점도 — 열입력에 따른 IR 열화상 실제 모습 (썸네일 {n_placed}장 / 전체 {len(plot_df)}건)',
             fontsize=16, fontweight='bold')
ax.legend(loc='upper left', fontsize=13, markerscale=1.5, prop={'weight': 'bold'})
ax.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "thermal_image_scatter.png"))
plt.show()

print(f"[완료] 썸네일 {n_placed}장 배치. 겹치거나 너무 작으면 N_THUMBS/IMG_ZOOM 값을 조정해 다시 실행하세요.")


## D-8 — 피팅된 물리 모델 파라미터 종합 시각화

지금까지 여러 셀에 흩어져 있던 물리 모델의 피팅 파라미터(Bad 하한 $Q_{min}$, 너겟 성장 $D_0$·$D_{max}$·$\tau$,
Explode 임계값)를 **오차막대와 함께 한 화면**에 모읍니다. 색으로 신뢰도를 바로 구분합니다 — 오차비율(오차/값)이
30% 미만이면 초록(신뢰 가능), 이상이면 빨강(불확실)입니다. 이 기준은 노트북 전체에서 일관되게 써온
`fit_reliability_ratio` 판정과 동일합니다.

**주의:** Bad 모델의 기울기 $k$와 `safe_heat_min`은 원래 피팅 단계에서 오차를 별도로 저장하지 않아 이
그래프에는 포함하지 않았습니다 (오차 전파를 하려면 `step3_defect_fit_params.csv` 저장 로직에 자코비안
계산을 추가해야 합니다 — 향후 과제).

**출력 파일:** `result_rsw/step16_fitted_parameters_summary.csv`, `result_rsw/figures/fitted_parameters_overview.png`


In [ ]:
# ==========================================
# [Cell 20] 피팅된 물리 모델 파라미터 종합 시각화
# ==========================================
defect_fit = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]
nugget_fit = pd.read_csv(os.path.join(RESULT_DIR, "step4_nugget_fit_params.csv")).iloc[0]

params = [
    {"name": "Q_min\n(Bad 하한)", "value": defect_fit["q_min_opt"], "err": defect_fit["q_min_err"],
     "reliability": defect_fit["fit_reliability_ratio"]},
    {"name": "D0\n(너겟 초기값, mm)", "value": nugget_fit["D0_opt"], "err": nugget_fit["D0_err"],
     "reliability": nugget_fit["D0_err"] / nugget_fit["D0_opt"]},
    {"name": "D_max\n(너겟 포화값, mm)", "value": nugget_fit["Dmax_opt"], "err": nugget_fit["Dmax_err"],
     "reliability": nugget_fit["Dmax_err"] / nugget_fit["Dmax_opt"]},
    {"name": "tau\n(너겟 시상수)", "value": nugget_fit["tau_opt"], "err": nugget_fit["tau_err"],
     "reliability": nugget_fit["tau_err"] / nugget_fit["tau_opt"]},
]

explode_fit_path = os.path.join(RESULT_DIR, "step11_explode_fit_params.csv")
if os.path.exists(explode_fit_path):
    explode_fit = pd.read_csv(explode_fit_path).iloc[0]
    if explode_fit["mode"] == "continuous":
        params.append({
            "name": f"X_mid\n(Explode, {explode_fit['driver']})",
            "value": explode_fit["x_mid_opt"], "err": explode_fit["x_mid_err"],
            "reliability": explode_fit["fit_reliability_ratio"],
        })
        params.append({
            "name": f"안전상한\n(Explode, {explode_fit['driver']})",
            "value": explode_fit["safe_max"], "err": explode_fit["safe_max_err"],
            "reliability": explode_fit["safe_max_err"] / abs(explode_fit["safe_max"]),
        })

n_params = len(params)
fig, axes = plt.subplots(1, n_params, figsize=(3.0 * n_params, 4.5), dpi=130)
if n_params == 1:
    axes = [axes]

for ax, p in zip(axes, params):
    reliable = p["reliability"] < 0.3
    color = '#2ca02c' if reliable else '#d62728'
    ax.errorbar([0], [p["value"]], yerr=[p["err"]], fmt='o', markersize=10,
                color=color, ecolor=color, capsize=8, elinewidth=2)
    ax.set_title(p["name"], fontsize=14, fontweight='bold')
    ax.set_xlim(-1, 1); ax.set_xticks([])
    verdict = "신뢰 가능" if reliable else "불확실"
    ax.text(0.5, 0.02, f"{p['value']:.3g} ± {p['err']:.3g}\n(오차비율 {p['reliability']*100:.0f}%, {verdict})",
            transform=ax.transAxes, fontsize=11, fontweight='bold', ha='center', va='bottom')
    ax.grid(True, axis='y', linestyle=':', alpha=0.4)

fig.suptitle('피팅된 물리 모델 파라미터 종합 (오차비율 <30% = 초록/신뢰 가능, ≥30% = 빨강/불확실)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "fitted_parameters_overview.png"))
plt.show()

param_summary_df = pd.DataFrame(params)
param_summary_df.to_csv(os.path.join(RESULT_DIR, "step16_fitted_parameters_summary.csv"), index=False)
display(param_summary_df)


## D-9 — 실시간 대응을 위한 2D 공정 위상도 (Phase Diagram)

지금까지의 그래프는 전부 **한 축(열입력 $Q$ 또는 전극각도)씩** 봤습니다. 실제 로봇이 다음 용접의
전류·통전시간을 실시간으로 정할 때는 **두 제어변수를 동시에** 놓고 어느 조합이 안전한지 한눈에
봐야 합니다. 여기서는 **전류 × 통전시간 평면**에 D-1(Bad 하한)·B-5(Explode 상한, 전극각도)의 피팅
결과를 겹쳐서, 실시간 룩업에 쓸 수 있는 위상도를 그립니다.

- **배경 색**: 빨강(Bad 위험, $Q < Q_{min}-오차$) / 노랑(경계, $Q_{min}-오차 \le Q < Q_{safe}$) /
  초록(안전, $Q \ge Q_{safe}$)
- **축 범위**: 전류 0~5,000A, 통전시간 0~2.0s로 고정 (관측 표본 범위가 아니라 로봇이 실제로 낼 수 있는 제어 범위 기준)
- **회색 등고선**: 열입력 $Q=I^2t$ 등고선 (참고용)
- **점**: 실제 샘플(Good/Bad/Explode)을 겹쳐 찍어 피팅 경계가 실측과 맞는지 검증
- **패널 2개**: Explode 요인(B-5에서 채택된 변수, 현재는 전극각도)의 두 수준을 나란히 비교 —
  한쪽은 Explode 안전, 다른 쪽은 Explode 위험 상승이라는 걸 패널 제목에 함께 표시

**한계:** 이 위상도는 열입력(Bad)과 전극각도(Explode)라는 **서로 다른 두 축**을 하나의 그림에
녹인 것이라, 두 결함이 완전히 독립이라는 가정 하에서만 유효합니다 (교호작용은 미검증 — F섹션
향후 과제 참고).

**출력 파일:** `result_rsw/figures/phase_diagram_2d.png`


In [ ]:
# ==========================================
# [D-9] 실시간 대응을 위한 2D 공정 위상도 (Phase Diagram)
# ==========================================
defect_fit_pd = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]
agg_pd = pd.read_csv(os.path.join(RESULT_DIR, "step1_aggregated_samples.csv"))

Q_min_pd = defect_fit_pd["q_min_opt"]
Q_min_err_pd = defect_fit_pd["q_min_err"]
Q_safe_pd = defect_fit_pd["safe_heat_min"]

PHASE_I_MAX = 5000.0   # 전류(A) 축 상한 -- 관측 범위가 아닌 로봇 제어 가능 범위 기준으로 고정
PHASE_T_MAX = 2.0      # 통전시간(s) 축 상한
I_grid = np.linspace(0.0, PHASE_I_MAX, 300)
t_grid = np.linspace(0.0, PHASE_T_MAX, 300)
II, TT = np.meshgrid(I_grid, t_grid)
QQ = II ** 2 * TT

# 0=위험(Bad) / 1=경계(불확도 구간) / 2=안전
zone = np.where(QQ < Q_min_pd - Q_min_err_pd, 0, np.where(QQ < Q_safe_pd, 1, 2))

explode_fit_path = os.path.join(RESULT_DIR, "step11_explode_fit_params.csv")
explode_fit_pd = pd.read_csv(explode_fit_path).iloc[0] if os.path.exists(explode_fit_path) else None
is_categorical_angle = (explode_fit_pd is not None
                         and explode_fit_pd["mode"] == "categorical"
                         and explode_fit_pd["driver"] == "angle_deg")

if is_categorical_angle:
    safe_levels_pd = [float(v) for v in str(explode_fit_pd["safe_levels"]).split(",") if v != ""]
    angle_levels = sorted(agg_pd["angle_deg"].unique())
else:
    angle_levels = [None]

cat_colors_pd = {'Good': '#2ca02c', 'Bad': '#d62728', 'Explode': '#ff7f0e'}
zone_cmap = plt.matplotlib.colors.ListedColormap(['#f8d7d7', '#ffe9a8', '#d5f2d5'])

fig, axes = plt.subplots(1, len(angle_levels), figsize=(7 * len(angle_levels), 6), dpi=120, squeeze=False)
axes = axes.flatten()

for ax, angle in zip(axes, angle_levels):
    ax.contourf(II, TT, zone, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap)
    cs = ax.contour(II, TT, QQ, levels=8, colors='gray', linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=9, fmt='%.0e')

    sub = agg_pd if angle is None else agg_pd[agg_pd["angle_deg"] == angle]
    for cat, color in cat_colors_pd.items():
        s = sub[sub["category"] == cat]
        ax.scatter(s["avg_current_A"], s["weld_time_s"], s=15, color=color, alpha=0.75,
                   edgecolor='black', linewidth=0.3, label=cat)

    title = "전체 샘플"
    if angle is not None:
        title = f"전극각도 {angle:g}°"
        if is_categorical_angle:
            title += "  (Explode 안전)" if angle in safe_levels_pd else "  (Explode 위험 ↑)"
    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("전류 Current (A)"); ax.set_ylabel("통전시간 Weld Time (s)")
    ax.set_xlim(0, PHASE_I_MAX); ax.set_ylim(0, PHASE_T_MAX)
    ax.legend(loc="upper right", fontsize=11, prop={'weight': 'bold'})

fig.suptitle("실시간 대응용 2D 공정 위상도  —  빨강(Bad 위험) / 노랑(경계) / 초록(안전)",
             fontsize=18, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "phase_diagram_2d.png"))
plt.show()

print(f"[안전 경계] Q_min={Q_min_pd:.3e}±{Q_min_err_pd:.3e}, Q_safe={Q_safe_pd:.3e}")
if is_categorical_angle:
    print(f"[Explode 안전 수준] {explode_fit_pd['driver']} in {safe_levels_pd}")


## D-10 — 실시간 종합 모니터링 대시보드 (2×2 애니메이션)

IR 이미지가 있는 99개 샘플을 **열입력 $Q$ 오름차순**으로 정렬해 재생하는 2×2 대시보드입니다.
낮은 열입력(위험)에서 높은 열입력(안전)으로 이동하며 4개 패널이 함께 움직입니다.

| 위치 | 내용 | 함께 보여주는 정보 |
|---|---|---|
| 1행1열 | IR 이미지 (왼쪽 상단에 흰색 텍스트로 샘플 라벨 표시) | 축척바(10mm)를 별도로 복원해 표시 |
| 1행2열 | D-9 공정 위상도, 전극각도에 따라 배경 자동 전환 | **환경**: 현재 전류·통전시간·압력·각도 값 |
| 2행1열 | D-1 Bad 확률 곡선 + 구간별 실측 Mean±SEM 에러바 + 현재 위치, $Q_{min}$ 오차구간 음영 | **신뢰도**: Bad 확률(%)과 피팅 신뢰도(오차비율) |
| 2행2열 | 열화상 `hot_area_fraction` 구간별 평균±SEM 에러바 + 선형 피팅선 + 현재 위치 | **품질 예측**: D-2 곡선 기반 예상 너겟지름, 그로부터 회귀선으로 역산한 예상 인장강도 |

이렇게 "이미지 → 환경 조건 → 결함 위험도(신뢰도 포함) → 예상 품질"이 한 화면에서 동시에 갱신되어,
개별 그래프를 따로 볼 때보다 실제 실시간 의사결정에 훨씬 가깝습니다.

**새로 추가한 계산**: 인장강도 예측은 D-5에서 상관계수($r=0.535$)만 확인했던 관계를 여기서 실제
`scipy.stats.linregress` 선형회귀선으로 만들어 사용합니다 (이 노트북에서 처음 만든 회귀식).

**한계:**
- 이미지 서브셋(99장)은 전체 열입력 범위의 하위 37%만 커버 — "안전 구역"쪽 절반은 이미지 없이
  위상도만 보여줍니다.
- Bad 확률 오차구간은 $Q_{min}$의 오차만 반영한 근사이며, $k$의 오차·공분산은 포함하지 않았습니다.
- 인장강도 예측은 단순 선형회귀이며 오차구간(신뢰구간)은 아직 표시하지 않습니다 — 향후 과제.

**재생 속도 조절**: `ANIM_INTERVAL_MS`(노트북 안에서 재생될 때 프레임 간 지연, ms 단위)와
`ANIM_FPS`(저장되는 GIF 파일 자체의 초당 프레임 수)로 조절합니다. **실제로 GIF 파일을 열어봤을 때의
속도를 좌우하는 건 `ANIM_FPS`**입니다 — 값을 키우면(예: 5→10) 두 배 빠르게, 줄이면(예: 5→2) 더
느리게 재생됩니다. `ANIM_INTERVAL_MS`는 Jupyter 안에서 애니메이션을 라이브로 볼 때만 영향을 줍니다.

**인터랙티브 재생**: GIF 저장과 별도로, matplotlib 내장 JS 플레이어(`to_jshtml()`)를 노트북에
표시합니다 — 재생/일시정지/처음·끝 이동/반복 버튼과 **프레임 슬라이더를 마우스로 드래그**해 원하는
지점으로 바로 이동할 수 있습니다. 별도 라이브러리 설치가 필요 없습니다.

**주의**: 프레임이 99장이라 인터랙티브 플레이어를 노트북에 표시하면 `.ipynb` 파일 용량이 꽤 커집니다
(프레임마다 PNG를 내장하므로 GIF보다 더 큼). 이후 이 노트북을 추가로 편집할 일이 있다면 출력을 지우고
작업하는 게 좋습니다. 용량이 부담되면 `ANIM_MAX_SAMPLES_D10`을 다시 정수로 지정해 프레임 수를 줄이세요.

**출력 파일:** `result_rsw/figures/image_phase_sync.gif`

In [ ]:
# ==========================================
# [D-10] 이미지-위상도-품질 종합 대시보드 애니메이션 (2x2)
# ==========================================
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import Patch
from scipy.stats import linregress

ANIM_MAX_SAMPLES_D10 = None  # None이면 IR 이미지가 있는 99개 샘플 전부 사용
# 프레임 수를 줄이고 싶으면 정수로 지정 (예: 40) -- 값을 줄일수록 렌더링/GIF 용량이 작아짐

# ---------- 예측 모델 준비 ----------
nugget_fit_pd = pd.read_csv(os.path.join(RESULT_DIR, "step4_nugget_fit_params.csv")).iloc[0]
k_bad_pd = defect_fit_pd["slope_k_opt"]

# 너겟지름 -> 인장강도: D-5에서 상관계수만 확인했던 관계를 여기서 실제 선형회귀선으로 사용
pull_lr = linregress(agg_pd["nugget_diameter_mm"], agg_pd["pull_test_N"])
print(f"[회귀] 인장강도 ≈ {pull_lr.slope:.1f} x 너겟지름 + {pull_lr.intercept:.1f}  (r={pull_lr.rvalue:.3f})")

def predict_bad_prob(Q):
    return ResistanceWeldingOptimizer.bad_rate_logistic_model(Q, Q_min_pd, k_bad_pd)

def predict_nugget(Q):
    return ResistanceWeldingOptimizer.nugget_growth_model(
        Q, nugget_fit_pd["D0_opt"], nugget_fit_pd["Dmax_opt"], nugget_fit_pd["tau_opt"])

def predict_pull_strength(nugget_d):
    return pull_lr.slope * nugget_d + pull_lr.intercept

# ---------- 애니메이션용 샘플·이미지 준비 ----------
anim_src = merged.sort_values("heat_input_proxy").reset_index(drop=True)
if ANIM_MAX_SAMPLES_D10 is not None and len(anim_src) > ANIM_MAX_SAMPLES_D10:
    idx_sel = np.linspace(0, len(anim_src) - 1, ANIM_MAX_SAMPLES_D10).astype(int)
    anim_src = anim_src.iloc[idx_sel].reset_index(drop=True)
print(f"[안내] 애니메이션에 사용할 샘플 수: {len(anim_src)}장")

anim_frames_d10 = []
for sid in anim_src["sample_id"]:
    fname = f"IR_{int(sid)}.jpg"
    img = plt.imread(os.path.join(IMG_DIR, fname))
    cropped, _ = crop_above_scalebar(img)
    anim_frames_d10.append(cropped)

def measure_scalebar_length_px(img_rgb: np.ndarray):
    """원본(크롭 전) 이미지에서 축척바 눈금선의 픽셀 길이를 측정한다 (이 길이가 10mm에 해당).
    B-4의 crop_above_scalebar와 별개로 동작하도록 독립적으로 다시 계산한다 — 이미 검증된
    B-4 함수는 그대로 두고, 여기서는 순수 조회 목적의 계산만 추가한다."""
    _, sat, val = to_hsv(img_rgb)
    white_mask = (sat < SAT_THRESHOLD) & (val > VAL_THRESHOLD)
    n_labels, labeled = cv2.connectedComponents(white_mask.astype(np.uint8), connectivity=8)
    best_w = None
    for label_id in range(1, n_labels):
        ys, xs = np.nonzero(labeled == label_id)
        bbox_w = xs.max() - xs.min() + 1
        if bbox_w < MIN_LINE_WIDTH_FRAC * img_rgb.shape[1]:
            continue
        if best_w is None or bbox_w > best_w:
            best_w = bbox_w
    return best_w

# 모든 이미지가 동일 카메라/렌즈이므로, 원본(크롭 전) 이미지 1장으로 축척(px/mm)을 한 번만 구해 재사용
sample_orig_img = plt.imread(os.path.join(IMG_DIR, f"IR_{int(anim_src.iloc[0]['sample_id'])}.jpg"))
scalebar_px_10mm = measure_scalebar_length_px(sample_orig_img)
print(f"[스케일 보정] 축척바 길이 {scalebar_px_10mm}px = 10mm")

# ---------- 2x2 Figure 구성 ----------
fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=105,
                          gridspec_kw={"wspace": 0.4, "hspace": 0.4})
ax_img_d10, ax_phase_d10 = axes[0, 0], axes[0, 1]
ax_bad_d10, ax_therm_d10 = axes[1, 0], axes[1, 1]

# --- 1행1열: IR 이미지 ---
im_artist_d10 = ax_img_d10.imshow(anim_frames_d10[0])
ax_img_d10.axis("off")
ax_img_d10.set_title("IR 열화상 이미지", fontsize=14, fontweight="bold")

# 원본 IR 이미지 자체가 이미 Jet 팔레트로 렌더링된 사진(절대 온도 계측값 아님)이므로,
# 컬러바는 실제 픽셀값이 아니라 Jet 색상표가 나타내는 상대적 열 강도(저->고)를 보여주는
# 참고용 범례다. fraction/pad를 작게 주어 imshow 이미지 자체의 원본 종횡비는 그대로 유지한다.
thermal_sm_d10 = plt.cm.ScalarMappable(cmap="jet", norm=plt.Normalize(vmin=0, vmax=1))
thermal_cbar_d10 = fig.colorbar(thermal_sm_d10, ax=ax_img_d10, fraction=0.046, pad=0.04)
thermal_cbar_d10.set_label("상대 열 강도 (Warmth)", fontsize=11, fontweight="bold")
thermal_cbar_d10.ax.tick_params(labelsize=9)
thermal_cbar_d10.set_ticks([0, 1])
thermal_cbar_d10.set_ticklabels(["낮음", "높음"])

if scalebar_px_10mm is not None:
    img_h, img_w = anim_frames_d10[0].shape[:2]
    y0 = img_h - img_h * 0.06  # 바닥 근처지만 막대 아래로 파란 배경이 보이게 살짝 띄움
    x0 = (img_w - scalebar_px_10mm) / 2.0   # 가로축 중앙 정렬
    x1 = x0 + scalebar_px_10mm
    ax_img_d10.plot([x0, x1], [y0, y0], color="white", lw=3, solid_capstyle="butt")
    ax_img_d10.text((x0 + x1) / 2.0, y0 - img_h * 0.04, "10mm",
                     color="white", fontsize=14, fontweight="bold", ha="center")

label_text_d10 = ax_img_d10.text(
    0.03, 0.97, "", transform=ax_img_d10.transAxes, color="white", fontsize=15,
    fontweight="bold", ha="left", va="top",
    bbox=dict(boxstyle="round", facecolor="black", alpha=0.4, pad=0.6))

# --- 1행2열: 위상도 (매 프레임 배경을 다시 그림 -- angle에 따라 안전/위험 패널이 바뀌므로) ---
def draw_phase_bg(ax, angle):
    ax.clear()
    ax.contourf(II, TT, zone, levels=[-0.5, 0.5, 1.5, 2.5], cmap=zone_cmap)
    cs = ax.contour(II, TT, QQ, levels=8, colors='gray', linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=9, fmt='%.0e')
    for cat, color in cat_colors_pd.items():
        s = agg_pd[(agg_pd["category"] == cat) & (agg_pd["angle_deg"] == angle)]
        ax.scatter(s["avg_current_A"], s["weld_time_s"], s=12, color=color, alpha=0.3)
    ax.set_xlabel("전류 Current (A)"); ax.set_ylabel("통전시간 Weld Time (s)")
    ax.set_xlim(0, PHASE_I_MAX); ax.set_ylim(0, PHASE_T_MAX)

    zone_legend_handles = [
        Patch(facecolor='#f8d7d7', edgecolor='gray', label='Bad 위험'),
        Patch(facecolor='#ffe9a8', edgecolor='gray', label='경계'),
        Patch(facecolor='#d5f2d5', edgecolor='gray', label='안전'),
    ]
    ax.legend(handles=zone_legend_handles, loc='center left', fontsize=11,
              title='위상 구분', title_fontsize=11, framealpha=0.9, prop={'weight': 'bold'})

# --- 2행1열: Bad 확률 곡선 (정적 배경 1회만 그림 -- Q_min 오차구간을 음영으로 함께 표시) ---
Q_line = np.linspace(agg_pd["heat_input_proxy"].min(), agg_pd["heat_input_proxy"].max() * 1.05, 300)
bad_curve = predict_bad_prob(Q_line)
bad_curve_lo = ResistanceWeldingOptimizer.bad_rate_logistic_model(Q_line, Q_min_pd - Q_min_err_pd, k_bad_pd)
bad_curve_hi = ResistanceWeldingOptimizer.bad_rate_logistic_model(Q_line, Q_min_pd + Q_min_err_pd, k_bad_pd)
ax_bad_d10.fill_between(Q_line, bad_curve_lo, bad_curve_hi, color="#2ca02c", alpha=0.15,
                         label="Q_min 오차구간")
ax_bad_d10.plot(Q_line, bad_curve, color="#2ca02c", lw=2, label="Bad 확률 곡선")
# D-1과 동일한 구간별 실측(Mean ± SEM) 에러바 -- 실제 피팅은 이 구간평균이 아니라 원본 493샘플
# 전체에 대해 이뤄졌지만(B-1c 참고), 어떤 실측 분포에서 이 곡선이 나왔는지 시각적으로 보여준다
ax_bad_d10.errorbar(binned["heat_mean"], binned["bad_rate"], yerr=binned["bad_rate_sem"],
                     fmt="o", color="#1f77b4", ecolor="#d62728", capsize=4, markersize=6,
                     label="구간별 실측 (Mean ± SEM)", zorder=3)
ax_bad_d10.set_xlabel("열입력 proxy Q"); ax_bad_d10.set_ylabel("Bad 확률")
ax_bad_d10.set_ylim(-0.05, 1.05)
ax_bad_d10.set_title("실시간 Bad 확률 + 신뢰도", fontsize=14, fontweight="bold")
ax_bad_d10.legend(fontsize=10, loc="upper right", prop={'weight': 'bold'})
bad_marker_d10, = ax_bad_d10.plot([], [], marker="*", markersize=20, color="black",
                                    markeredgecolor="yellow", markeredgewidth=1.2, zorder=5)
bad_text_d10 = ax_bad_d10.text(0.98, 0.62, "", transform=ax_bad_d10.transAxes, fontsize=11,
                                 fontweight="bold", va="top", ha="right",
                                 bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, pad=0.6))

# --- 2행2열: 열화상 고온영역 비율 -- 구간별 평균±SEM 에러바 + 선형 피팅선 ---
# (기존엔 개별 샘플 산점도만 있었는데, 추세가 피팅선처럼 보인다는 지적을 반영해
#  실제로 구간화한 평균±SEM 에러바와, 그 피팅에 쓰인 선형회귀선을 함께 표시한다)
therm_bins_d10 = pd.qcut(merged["heat_input_proxy"], q=6, duplicates="drop")
therm_binned_d10 = merged.groupby(therm_bins_d10, observed=True).agg(
    q_mean=("heat_input_proxy", "mean"),
    hot_mean=("hot_area_fraction", "mean"),
    hot_sem=("hot_area_fraction", "sem"),
    n=("hot_area_fraction", "size"),
).reset_index(drop=True)

therm_lr_d10 = linregress(merged["heat_input_proxy"], merged["hot_area_fraction"])
Q_therm_line = np.linspace(merged["heat_input_proxy"].min(), merged["heat_input_proxy"].max(), 200)
therm_fit_line = therm_lr_d10.slope * Q_therm_line + therm_lr_d10.intercept

ax_therm_d10.scatter(merged["heat_input_proxy"], merged["hot_area_fraction"], s=10,
                      color="#1f77b4", alpha=0.15, label="개별 샘플(99건)")
ax_therm_d10.errorbar(therm_binned_d10["q_mean"], therm_binned_d10["hot_mean"],
                       yerr=therm_binned_d10["hot_sem"], fmt="o", color="#1f77b4",
                       ecolor="#d62728", capsize=4, markersize=6, label="구간별 평균 ± SEM")
ax_therm_d10.plot(Q_therm_line, therm_fit_line, color="#ff7f0e", lw=2,
                   label=f"선형 피팅 (r={therm_lr_d10.rvalue:.2f})")
ax_therm_d10.set_xlabel("열입력 proxy Q"); ax_therm_d10.set_ylabel("고온영역 비율 (hot_area_fraction)")
ax_therm_d10.set_title("열화상 기반 품질 예측", fontsize=14, fontweight="bold")
ax_therm_d10.legend(fontsize=10, loc="upper left", prop={'weight': 'bold'})
therm_marker_d10, = ax_therm_d10.plot([], [], marker="*", markersize=20, color="black",
                                        markeredgecolor="yellow", markeredgewidth=1.2, zorder=5)
therm_text_d10 = ax_therm_d10.text(0.02, 0.6, "", transform=ax_therm_d10.transAxes, fontsize=11,
                                     fontweight="bold", va="center", ha="left",
                                     bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, pad=0.6))

fig.suptitle("RSW 실시간 종합 모니터링 대시보드", fontsize=19, fontweight="bold")


def update_d10(i):
    row = anim_src.iloc[i]
    Q = row["heat_input_proxy"]

    # 1행1열
    im_artist_d10.set_data(anim_frames_d10[i])
    label_text_d10.set_text(f"Sample #{int(row['sample_id'])}")
    ax_img_d10.set_title(f"IR 열화상 이미지 — {row['category']}", fontsize=14, fontweight="bold")

    # 1행2열 (draw_phase_bg가 ax.clear()를 하므로 점·텍스트도 매번 다시 그림)
    draw_phase_bg(ax_phase_d10, row["angle_deg"])
    ax_phase_d10.scatter([row["avg_current_A"]], [row["weld_time_s"]], s=220, marker="*",
                          color="black", edgecolor="yellow", linewidth=1.5, zorder=5)
    explode_tag = ""
    if is_categorical_angle:
        explode_tag = "안전" if row["angle_deg"] in safe_levels_pd else "위험 ↑"
    ax_phase_d10.set_title(f"공정 위상도 — 전극각도 {row['angle_deg']:g}° (Explode {explode_tag})",
                            fontsize=14, fontweight="bold")
    ax_phase_d10.text(
        0.98, 0.02,
        f"전류 {row['avg_current_A']:.0f}A · 시간 {row['weld_time_s']:.2f}s\n"
        f"압력 {row['pressure_psi']:.0f}PSI · 각도 {row['angle_deg']:g}°",
        transform=ax_phase_d10.transAxes, fontsize=11, fontweight="bold", va="bottom", ha="right",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, pad=0.6))

    # 2행1열
    bad_p = predict_bad_prob(Q)
    bad_marker_d10.set_data([Q], [bad_p])
    reliability = defect_fit_pd["fit_reliability_ratio"]
    verdict = "신뢰 가능" if reliability < 0.3 else "주의(오차↑)"
    bad_text_d10.set_text(f"Bad 확률: {bad_p * 100:.1f}%\n피팅 신뢰도: {verdict} ({reliability * 100:.0f}%)")

    # 2행2열
    nugget_pred = predict_nugget(Q)
    pull_pred = predict_pull_strength(nugget_pred)
    therm_marker_d10.set_data([Q], [row["hot_area_fraction"]])
    therm_text_d10.set_text(
        f"예상 너겟지름: {nugget_pred:.2f}mm\n"
        f"예상 인장강도: {pull_pred:.0f}N (r={pull_lr.rvalue:.2f})")

    return im_artist_d10, label_text_d10, bad_marker_d10, bad_text_d10, therm_marker_d10, therm_text_d10


# 속도 조절: ANIM_INTERVAL_MS(노트북 안에서 재생될 때 프레임 간 지연, ms) /
# ANIM_FPS(저장되는 GIF 파일 자체의 초당 프레임 수 -- 실제 재생 속도는 이 값이 좌우함)
ANIM_INTERVAL_MS = 200  # 값을 줄이면(예: 100) 더 빠르게, 늘리면(예: 400) 더 느리게
ANIM_FPS = 5            # 저장된 GIF의 실제 재생 속도. 값을 늘리면(예: 10) GIF가 더 빠르게 재생됨

anim_d10 = FuncAnimation(fig, update_d10, frames=len(anim_frames_d10),
                          interval=ANIM_INTERVAL_MS, blit=False)

gif_path_d10 = os.path.join(FIGURE_DIR, "image_phase_sync.gif")
anim_d10.save(gif_path_d10, writer=PillowWriter(fps=ANIM_FPS))
print(f"[완료] {len(anim_frames_d10)}프레임 2x2 종합 대시보드 저장(GIF): {gif_path_d10}")

# 재생 / 일시정지 / 처음·끝으로 이동 / 프레임 슬라이더 드래그 탐색이 되는 인터랙티브 플레이어.
# matplotlib에 내장된 JS 애니메이션 플레이어(to_jshtml)를 쓰면 별도 라이브러리 없이 바로 된다.
# 프레임이 많으면(99장) 기본 임베드 용량 제한(20MB)을 넘을 수 있어 미리 늘려둔다.
plt.rcParams["animation.embed_limit"] = 100  # MB
from IPython.display import HTML, display as ipy_display

interactive_player_d10 = HTML(anim_d10.to_jshtml())
plt.close(fig)  # 정적 figure는 숨기고 인터랙티브 플레이어만 노트북에 표시
ipy_display(interactive_player_d10)
print("[안내] 위 플레이어의 재생(▶)/일시정지(⏸)/처음(⏮)·끝(⏭)/반복(Loop) 버튼과 "
      "프레임 슬라이더를 마우스로 드래그해 원하는 지점으로 바로 이동할 수 있습니다.")


# E. 결과 정리 및 분석

### 기(起) — 지금까지 무엇을 했는가

B 섹션에서는 물리 모델(Bad 하한, 너겟 성장, Explode 상한)과 딥러닝 대리모델, 열화상 파이프라인,
가드레일, 온라인 학습 시뮬레이션까지 총 6개의 서로 다른 파이프라인을 각각 독립적으로 구현했고,
C·D 섹션에서는 원자료 분포와 각 파이프라인의 피팅 결과를 개별 그래프로 확인했습니다. 여기 E 섹션은
그 결과들을 한곳에 모아 놓고 "그래서 결론이 무엇인가"에 답하는 자리입니다. **이 노트북의 실질적인
결론은 이 섹션을 기준으로 읽어주세요.**


## E-1 — 최종 종합 해석: 전체 파이프라인 통합

### 승(承) — 무엇을 종합하는가

물리 피팅(Bad 하한·너겟 성장), DL 대리모델 교차검증, 열화상 상관분석, Explode 상한 분석, 서두 상관계수
검증, 통합 가드레일까지 — 섹션 B~D의 모든 수치 결과를 아래 코드 셀 하나에서 다시 읽어와 한 화면에
출력하고, 동시에 텍스트 리포트(`step14_final_summary.txt`)로도 저장합니다.

### 전(轉) — 핵심 결과와 그 의미

수치만 나열하면 맥락을 놓치기 쉬우므로, 무엇을 확인했고 그것이 왜 중요한지 먼저 정리합니다.

- **미융착(Bad) 하한이 안정적으로 추정되었습니다.** 임계 열입력 $Q_{min}=(1.171\pm0.055)\times10^{6}$
  (오차비율 4.7%)로, 처음 시도했던 구간평균 피팅(오차비율 87배, 사실상 무의미)과 비교하면 원본 샘플
  전체에 직접 피팅하는 방식이 왜 필요했는지를 뒷받침하는 결과입니다. 이로부터 역산한 안전 열입력 하한은
  $2.409\times10^{6}$ 입니다.
- **너겟 성장 곡선은 절반만 신뢰할 수 있습니다.** 포화값 $D_{max}=4.13\pm0.61$mm(오차비율 15%,
  신뢰 가능)는 안정적이지만, 성장 시상수 $\tau$는 오차비율이 108%로 사실상 미결정입니다. 관측 구간
  내에서 성장이 거의 선형으로 보여 포화 시점 자체를 데이터가 특정하지 못하기 때문입니다 — "얼마나
  크게 자라는가"는 알 수 있어도 "얼마나 빨리 포화되는가"는 이 데이터만으로 답하기 어렵다는 뜻입니다.
- **Explode(팽출)는 예상과 달리 전극각도와 관계가 있었습니다.** 서두에 인용된 "열입력과 무관"이라는
  설명과 달리, 5개 후보 변수 중 전극각도가 가장 강한 단변량 관계를 보였습니다($r=0.144$, $p=0.0014$).
  다만 전극각도는 0°/15° 두 수준뿐인 범주형에 가까워, 안전 상한 $2.52\pm3.04°$는 오차가 추정값보다
  커 정량적 임계값이 아니라 "0°는 안전, 15°는 위험 증가"라는 정성적 결론으로만 해석해야 합니다.
- **딥러닝 대리모델은 층화 교차검증으로 신뢰도를 확보했습니다.** 단일 80/20 분할의 F1=0.857은 Bad
  검증 표본이 4건뿐이라 신뢰구간이 넓었지만, Stratified 5-Fold로 493건 전체를 정확히 한 번씩 검증한
  결과 F1=0.892±0.062(풀링 혼동행렬 TP17·FP0·FN4·TN472)로 더 안정적인 추정치를 얻었습니다.
- **열화상 물리 특징은 통계적으로 유의하지만 대표성엔 한계가 있습니다.** 고온영역 비율과 열입력의
  상관($r=0.410$, $p<0.0001$)을 비롯해 3개 관계 모두 유의했지만, IR 이미지가 있는 99개 표본은 전체
  열입력 범위의 하위 37%만 커버하는 비무작위 부분집합이므로 탐색적 결과로 한정해 해석합니다.
- **서두에 인용했던 상관계수 3개는 모두 실측으로 재현되었습니다.** 통전시간↔너겟지름(인용 0.47,
  실측 0.473), 열입력↔너겟지름(인용 0.36, 실측 0.352), 너겟지름↔인장강도(인용 0.53, 실측 0.535) —
  "검증되지 않은 채 인용된 숫자"를 코드로 직접 재현했다는 점에서, 이 노트북이 표방하는 재현성 원칙이
  실제로 지켜졌음을 보여줍니다.

아래 코드 셀은 위 요약의 근거가 되는 원본 수치를 CSV에서 다시 불러와 표와 함께 출력합니다.


In [ ]:
# ==========================================
# [Cell 16] 최종 종합 해석 (+ 텍스트 리포트 저장)
# ==========================================
report_lines = []
def emit(text=""):
    report_lines.append(text)
    print(text)

sep = "=" * 66

emit(sep); emit("  A. 서두 상관계수 주장 검증"); emit(sep)
claim_df = pd.read_csv(os.path.join(RESULT_DIR, "step9_intro_claims_check.csv"))
display(claim_df)
report_lines.append(claim_df.to_string(index=False))

emit(); emit(sep); emit("  B. 미융착(Bad) 하한 + 너겟 성장"); emit(sep)
d = pd.read_csv(os.path.join(RESULT_DIR, "step3_defect_fit_params.csv")).iloc[0]
n = pd.read_csv(os.path.join(RESULT_DIR, "step4_nugget_fit_params.csv")).iloc[0]
emit(f"  안전 열입력 하한: {d['safe_heat_min']:.3e}  (Q_min 오차비율 {d['fit_reliability_ratio']:.2f})")
emit(f"  D_max = {n['Dmax_opt']:.2f} ± {n['Dmax_err']:.2f} mm  "
     f"(tau 오차비율 {n['tau_err']/n['tau_opt']:.2f} — 사실상 미결정, 참고용)")

emit(); emit(sep); emit("  C. Explode 상한"); emit(sep)
if EXPLODE_DRIVER is not None:
    e = pd.read_csv(os.path.join(RESULT_DIR, "step11_explode_fit_params.csv")).iloc[0]
    if e['mode'] == 'categorical':
        emit(f"  기준 변수: {e['driver']} (범주형)  안전 수준: {e['safe_levels']}")
    else:
        emit(f"  기준 변수: {e['driver']}  안전 상한: {e['safe_max']:.3e}  "
             f"(오차비율 {e['fit_reliability_ratio']:.2f})")
else:
    emit("  단변량으로 유의미한 요인을 찾지 못함 (다변량 분석이 향후 과제)")

emit(); emit(sep); emit("  D. 딥러닝 대리모델 (Stratified 5-Fold)"); emit(sep)
cvs = pd.read_csv(os.path.join(RESULT_DIR, "step12b_cv_summary.csv")).iloc[0]
emit(f"  Fold F1 평균±표준편차: {cvs['f1_mean']:.3f} ± {cvs['f1_std']:.3f}  "
     f"(기존 단일분할 F1=0.857과 비교)")
emit(f"  풀링 혼동행렬: TP={cvs['pooled_tp']} FP={cvs['pooled_fp']} "
     f"FN={cvs['pooled_fn']} TN={cvs['pooled_tn']}")

emit(); emit(sep); emit("  E. 열화상(IR) 물리 특징 상관분석"); emit(sep)
thermal_corr_df = pd.read_csv(os.path.join(RESULT_DIR, "step8_thermal_correlations.csv"))
display(thermal_corr_df)
report_lines.append(thermal_corr_df.to_string(index=False))
emit("  주의: 이미지 서브셋은 전체 열입력 범위의 하위 37%만 커버 (선택 편향 — 탐색적 결과로만 활용)")

emit(); emit(sep); emit("  F. 완전한 공정 윈도우 가드레일"); emit(sep)
guard_df = pd.read_csv(os.path.join(RESULT_DIR, "step13_full_guardrail_log.csv"))
display(guard_df)
report_lines.append(guard_df.to_string(index=False))

summary_path = os.path.join(RESULT_DIR, "step14_final_summary.txt")
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))
print(f"\n[저장 완료] 최종 종합 해석 전체 텍스트를 {summary_path} 에 저장했습니다.")


# F. 트러블슈팅 및 개선 방안

### 결(結) — 결론과 남은 과제

여기서부터는 이 노트북을 만드는 과정에서 실제로 겪은 시행착오를 정리합니다. 앞서 E 섹션에서 제시한
결론들은 결과만 놓고 보면 매끄러워 보이지만, 그 결론에 이르기까지 여러 차례 잘못된 접근을 시도하고
수정하는 과정을 거쳤습니다. 이 과정 자체를 투명하게 남기는 것이 "왜 이 방법을 선택했는가"에 대한
가장 설득력 있는 답이라고 판단해, 발견 순서대로 표로 정리했습니다. 각 항목의 상세한 코드 수준 설명은
해당 구현 셀(B 섹션)의 마크다운에도 남아 있습니다 — 여기서는 전체를 한눈에 보는 색인입니다.


## F-1 — 발견·수정한 버그 전체 목록

| # | 위치 | 증상 | 원인 | 해결 |
|---|---|---|---|---|
| 1 | B-1c (Bad 하한) | $Q_{min}$ 오차비율 87배 발산 | 열입력 10구간 중 신호 있는 구간이 1개뿐 → 구간평균 피팅이 미결정 | 구간화 이전 원본 493샘플에 직접 피팅 + bounds 추가 → 오차비율 4.7%로 개선 |
| 2 | B-1c (너겟 성장) | $D_{max}=688$mm, $\tau=2.3\times10^{10}$로 발산 | bounds 없는 피팅이 비물리적 해로 수렴 | 물리적으로 타당한 bounds(관측 최댓값 기준) 추가 |
| 3 | B-1c (구간화) | 표본 1개 구간의 SEM=0 | 등간격 구간화로 극단 구간에 표본이 몰림 | `qcut` 균등분할 + Agresti-Coull 보정 SEM |
| 4 | B-4 (열화상) | 밝기 기반 "가장 뜨거운 영역"이 실제 온도와 무관 | Jet 컬러맵은 밝기가 온도와 비단조 대응 + 흰색 오버레이(십자선·축척바)가 항상 최댓값 근처 | Hue 기반 warmth로 재설계 + 채도 마스킹으로 오버레이 제외 |
| 5 | B-4 (열화상) | 축척바가 "가장 뜨거운 영역"으로 오검출 | 흰색 눈금선+텍스트가 채도 낮고 밝기 높음 | 연결영역 분석으로 눈금선 검출 후 그 위쪽만 크롭 (선이 기울어져도 최상단 행 기준으로 대응) |
| 6 | B-4 (열화상) | `hotspot_aspect_ratio`가 최대 54.3까지 발산 | 노이즈성 소블롭이 최대 연결영역으로 잡힘 | `MIN_BLOB_PIXELS` 가드 추가 (단, 이것만으론 부족했음 — 근본 원인은 오버레이 자체) |
| 7 | B-4 (열화상) | 십자선의 검은 테두리가 채도 필터를 통과할 위험 | JPEG 압축의 색 번짐으로 어두운 픽셀에 약간의 채도가 남을 수 있음 | 밝기 하한(`MIN_VALID_VALUE=0.2`)을 채도 조건과 대칭으로 추가 |
| 8 | B-5 (Explode) | 구간별 실측 점이 1개로 붕괴 | `angle_deg`(0/15 두 값뿐)에 연속형용 `pd.qcut(q=8)` 적용 → 분위수 경계가 겹쳐 `duplicates="drop"`이 전부 제거 | 고유값 10개 이하면 실제 값별로 직접 집계 |
| 9 | B-5 (Explode) | 피팅 곡선이 실측점(9.8%)과 5배 어긋남(예측 50%) | 범주형(2값) 변수에 연속 로지스틱을 강제 피팅 — $X_{mid}$·$k$가 수학적으로 유일하게 결정되지 않아 경계값에서 멈춤 | 범주형/연속형 분기 설계 — 범주형은 로지스틱 대신 수준별 실측 비율 직접 비교 |
| 10 | B-2 (DL 학습) | val_loss가 epoch 9 이후 계속 상승(과적합) | `CosineAnnealingLR(T_max=NUM_EPOCHS=100)`인데 epoch 9 시점 LR이 초기값의 98%로 거의 감쇠 안 됨 | `NUM_EPOCHS`를 20으로 줄여 같은 스케줄이 실제로 감쇠하도록 조정 |
| 11 | B-2 (DL 평가) | 단일 80/20 분할 F1=0.857의 신뢰도 불명 | Bad 검증표본이 약 4개뿐이라 통계적으로 불안정 | Stratified 5-Fold CV 도입 (F1=0.892±0.062, 493건 전체를 정확히 한 번씩 검증) |
| 12 | D-3 (학습이력 그래프) | 범례 누락, x축 비정수 가능 | `twinx()` 두 축에 `legend()` 미호출 | handle 병합 + `MaxNLocator(integer=True)` |
| 13 | A (서두 주장) | 상관계수 인용값이 코드로 검증된 적 없음 | 사전 검증 결과를 텍스트로만 기재 | D-5에서 실제 `pearsonr` 계산 후 대조 (3건 모두 일치 확인) |

## F-2 — 구조적 한계 (완전히 해결하지 못한 것들)

| 한계 | 내용 |
|---|---|
| Bad 모델 오차 전파 불완전 | `slope_k`와 `safe_heat_min` 자체의 오차가 저장되지 않아 D-1 그래프에 반영 안 됨 |
| 너겟 성장 시상수 $\tau$ 미결정 | 관측 구간 내 성장이 근사적으로 선형이라 오차비율 108% — 참고용으로만 사용 |
| Explode 단변량 분석의 한계 | 5개 후보 변수를 개별적으로만 검정 — 다변량 상호작용은 미탐색 |
| IR 이미지 표본 편향 | Sample ID 10~18·100~189 구간에 몰려 있어 전체 열입력 범위의 하위 37%만 커버 (공개 데이터셋 자체의 한계로 추가 수집 불가) |
| Phase 1 온라인 시뮬레이션의 근사 | 진짜 순차 베이지안 갱신이 아니라 배치 단위 재적합 근사 |
| D-9 위상도의 독립성 가정 | 열입력(Bad)과 전극각도(Explode)를 별개 축으로 겹쳐 그렸을 뿐, 두 결함의 교호작용은 검증하지 않음 |

## F-3 — 향후 개선 과제

1. `step3_defect_fit_params.csv`에 $k$·`safe_heat_min`의 자코비안 기반 오차를 추가 저장
2. Explode 다변량 로지스틱 회귀(예: `angle_deg` + `pressure_psi` 상호작용) 시도
3. IR 이미지 라디오메트릭 캘리브레이션이 가능한 데이터셋으로 교체 시 절대 온도 기반 재분석
4. Phase 1을 실제 순차 베이지안 갱신(또는 칼만 필터)으로 고도화
5. D-9 위상도를 Bad·Explode 결합 다차원 안전영역으로 확장하는 최적화(베이지안 최적화/RL) 도입

## F-4 — 결론적으로

이 노트북은 "물리 법칙 기반 곡선 피팅 + 오차비율에 의한 신뢰도 판정"이라는 하나의 원칙을 결함
메커니즘이 다른 두 문제(Bad 하한, Explode 상한)와 두 종류의 데이터(정형 수치, IR 이미지)에 일관되게
적용했습니다. 그 결과 일부 추정치(Q_min, D_max, CV F1)는 실무에 바로 참고할 수 있을 만큼 신뢰도가
높은 반면, 다른 일부(성장 시상수 $\tau$, Explode 정량 임계값)는 현재 데이터의 한계로 신뢰구간이 넓어
"참고용"으로만 남았습니다. 이렇게 **신뢰할 수 있는 결과와 그렇지 못한 결과를 같은 기준(오차비율 30%)
으로 나누어 명시**한 것이 이 파이프라인의 핵심 기여이며, 향후 개선 과제 역시 신뢰도가 낮았던 지점을
중심으로 정리했습니다.
